In [0]:
import sys, os, subprocess

DBT_LIB = "/tmp/dbtlib"
if not os.path.exists(f"{DBT_LIB}/dbt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--ignore-installed",
                    "--target", DBT_LIB, "dbt-databricks>=1.8"], check=True)

env_check = dict(os.environ, PYTHONPATH=DBT_LIB, PYTHONNOUSERSITE="1")
r = subprocess.run([sys.executable, "-c", "from dbt.cli.main import cli; cli()", "--version"],
                   capture_output=True, text=True, env=env_check)
print(r.stdout or r.stderr[-2000:])

In [0]:
import os

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"
for d in ["models/staging", "models/intermediate", "models/marts", "macros", "snapshots", "seeds", "tests"]:
    os.makedirs(f"{DBT}/{d}", exist_ok=True)

files = {
"dbt_project.yml": """name: oil_shock
version: "1.0.0"
config-version: 2
profile: oil_shock

model-paths: ["models"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]
seed-paths: ["seeds"]
test-paths: ["tests"]

models:
  oil_shock:
    staging:
      +schema: silver
      +materialized: view
    intermediate:
      +schema: silver
      +materialized: view
    marts:
      +schema: gold
      +materialized: table
""",
"macros/generate_schema_name.sql": """{% macro generate_schema_name(custom_schema_name, node) -%}
  {%- if custom_schema_name is none -%}{{ target.schema }}{%- else -%}{{ custom_schema_name | trim }}{%- endif -%}
{%- endmacro %}
""",
".gitignore": "target/\nlogs/\ndbt_packages/\n",
}

for path, content in files.items():
    with open(f"{DBT}/{path}", "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
import os, sys, subprocess

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"
DBT_LIB = "/tmp/dbtlib"
PROFILES = "/tmp/dbt_profiles"
os.makedirs(PROFILES, exist_ok=True)

with open(f"{PROFILES}/profiles.yml", "w") as f:
    f.write("""oil_shock:
  target: prod
  outputs:
    prod:
      type: databricks
      catalog: oil_shock
      schema: silver
      host: dbc-050dc881-f83e.cloud.databricks.com
      http_path: /sql/1.0/warehouses/42944d45ee999e1c
      token: "{{ env_var('DBT_DATABRICKS_TOKEN') }}"
      threads: 4
""")

env = dict(os.environ,
           PYTHONPATH=DBT_LIB, PYTHONNOUSERSITE="1",
           DBT_DATABRICKS_TOKEN=dbutils.secrets.get("oil-shock", "databricks-token"),
           DBT_TARGET_PATH="/tmp/dbt_target",
           DBT_LOG_PATH="/tmp/dbt_logs",
           DBT_PROFILES_DIR=PROFILES)

def run_dbt(*args):
    r = subprocess.run([sys.executable, "-c", "from dbt.cli.main import cli; cli()",
                        *args, "--project-dir", DBT],
                       capture_output=True, text=True, env=env)
    print(r.stdout[-6000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
    print("SUCCESS:", r.returncode == 0)
    return r.returncode == 0

run_dbt("debug")

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/staging/_sources.yml": """version: 2

sources:
  - name: bronze
    database: oil_shock
    schema: bronze
    description: Raw data as downloaded by the Python loaders (phase 1).
    tables:
      - name: eia_series
        description: EIA API v2 series values (prices, US balances, rigs, STEO forecasts by release month).
      - name: jodi_oil
        description: JODI-Oil World Database monthly balances by country, product and flow.
      - name: portwatch_chokepoints
        description: IMF PortWatch daily chokepoint transits. Sources UN Global Platform; IMF PortWatch.
      - name: cftc_cot
        description: CFTC disaggregated futures-only Commitments of Traders (NYMEX WTI, RBOB, ULSD).
      - name: ice_cot
        description: ICE Futures Europe Commitments of Traders (Brent, gasoil, Dubai 1st Line).
      - name: fred_observations
        description: FRED series observations (dollar, rates, CPI, industry, VIX, IMF oil prices).
      - name: fred_series_meta
        description: FRED series descriptions incl. copyright notes.
      - name: oecd_cli
        description: OECD composite leading indicators by area, monthly.
      - name: noaa_hurdat2
        description: NOAA HURDAT2 Atlantic storm tracks, 6-hourly, 1990 onward.
      - name: noaa_active_storms
        description: Snapshots of NOAA active storms, one row per storm per run.
      - name: gdelt_events
        description: Oil-relevant GDELT events, filtered at ingestion. Source The GDELT Project.
""",
"models/staging/stg_eia__series.sql": """with src as (
    select * from {{ source('bronze', 'eia_series') }}
)

select
    concat_ws('|', series_id, period, coalesce(vintage, '')) as row_key,
    series_id,
    series_label,
    route,
    frequency,
    case
        when frequency = 'monthly' then to_date(concat(period, '-01'))
        else to_date(period)
    end                                             as period_date,
    period                                          as period_raw,
    value,
    units,
    nullif(vintage, '')                             as steo_vintage,
    case when route = 'steo' then 'forecast' else 'actual' end as data_kind,
    loaded_at
from src
where value is not null
""",
"models/staging/stg_eia__series.yml": """version: 2

models:
  - name: stg_eia__series
    description: EIA series values with proper dates; actual data vs STEO forecasts (by release month).
    columns:
      - name: row_key
        description: series_id | period | vintage
        tests: [unique, not_null]
      - name: series_id
        tests: [not_null]
      - name: period_date
        tests: [not_null]
      - name: value
        tests: [not_null]
      - name: frequency
        tests:
          - accepted_values:
              arguments:
                values: ['daily', 'weekly', 'monthly']
      - name: data_kind
        tests:
          - accepted_values:
              arguments:
                values: ['actual', 'forecast']
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "stg_eia__series")

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/staging/stg_jodi__oil.sql": """select
    concat_ws('|', category, ref_area, time_period, energy_product, flow_breakdown, unit_measure) as row_key,
    category,
    ref_area                                as country_code,
    to_date(concat(time_period, '-01'))     as period_date,
    energy_product,
    flow_breakdown,
    unit_measure,
    obs_value,
    obs_value is not null                   as is_reported,
    assessment_code,
    source_file,
    loaded_at
from {{ source('bronze', 'jodi_oil') }}
""",
"models/staging/stg_portwatch__chokepoints.sql": """select
    concat_ws('|', portid, date)                                         as row_key,
    to_date(date)                                                        as transit_date,
    portid                                                               as chokepoint_id,
    portname                                                             as chokepoint_name,
    cast(get_json_object(attributes_json, '$.n_total') as int)           as ships_total,
    cast(get_json_object(attributes_json, '$.n_tanker') as int)          as ships_tanker,
    cast(get_json_object(attributes_json, '$.n_container') as int)       as ships_container,
    cast(get_json_object(attributes_json, '$.n_dry_bulk') as int)        as ships_dry_bulk,
    cast(get_json_object(attributes_json, '$.n_general_cargo') as int)   as ships_general_cargo,
    cast(get_json_object(attributes_json, '$.n_roro') as int)            as ships_roro,
    cast(get_json_object(attributes_json, '$.capacity') as double)       as capacity_total_mt,
    cast(get_json_object(attributes_json, '$.capacity_tanker') as double) as capacity_tanker_mt,
    loaded_at
from {{ source('bronze', 'portwatch_chokepoints') }}
""",
"models/staging/stg_fred__observations.sql": """select
    concat_ws('|', o.series_id, o.date)    as row_key,
    o.series_id,
    m.title                                as series_title,
    m.units,
    m.frequency,
    to_date(o.date)                        as obs_date,
    o.value,
    m.has_copyright_note,
    o.loaded_at
from {{ source('bronze', 'fred_observations') }} o
left join {{ source('bronze', 'fred_series_meta') }} m
    on o.series_id = m.series_id
where o.value is not null
""",
"models/staging/stg_oecd__cli.sql": """select
    concat_ws('|', ref_area, time_period)  as row_key,
    ref_area                               as area_code,
    area_name,
    to_date(concat(time_period, '-01'))    as period_date,
    obs_value                              as cli_value,
    loaded_at
from {{ source('bronze', 'oecd_cli') }}
where obs_value is not null
""",
"models/staging/stg_noaa__storm_tracks.sql": """select
    concat_ws('|', storm_id, obs_time)     as row_key,
    storm_id,
    name                                   as storm_name,
    to_timestamp(obs_time)                 as obs_ts,
    year(to_timestamp(obs_time))           as season,
    record_id,
    status,
    latitude,
    longitude,
    max_wind_kt,
    min_pressure_mb,
    source_file,
    loaded_at
from {{ source('bronze', 'noaa_hurdat2') }}
""",
"models/staging/stg_noaa__active_storms.sql": """select
    snapshot_at,
    storm_id,
    name                                   as storm_name,
    classification,
    intensity_kt,
    pressure_mb,
    latitude,
    longitude,
    last_update
from {{ source('bronze', 'noaa_active_storms') }}
""",
"models/staging/stg_gdelt__events.sql": """select
    globaleventid                                          as event_id,
    to_date(event_date, 'yyyyMMdd')                        as event_date,
    to_date(substr(date_added, 1, 8), 'yyyyMMdd')          as news_date,
    case when length(date_added) = 14
         then to_timestamp(date_added, 'yyyyMMddHHmmss') end as news_ts,
    datediff(to_date(substr(date_added, 1, 8), 'yyyyMMdd'),
             to_date(event_date, 'yyyyMMdd'))              as days_event_before_news,
    actor1_code, actor1_name, actor1_country,
    actor2_code, actor2_name, actor2_country,
    is_root_event,
    event_code, event_base_code, event_root_code,
    quad_class, goldstein,
    num_mentions, num_sources, num_articles, avg_tone,
    geo_type, geo_name, geo_country, geo_lat, geo_long,
    source_url,
    matched_by,
    source_file,
    loaded_at
from {{ source('bronze', 'gdelt_events') }}
""",
"models/staging/_staging.yml": """version: 2

models:
  - name: stg_jodi__oil
    description: JODI monthly balances by country, product, flow and unit.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: period_date
        tests: [not_null]
  - name: stg_portwatch__chokepoints
    description: Daily ship transits per chokepoint, by vessel type, with tanker capacity.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: ships_total
        tests: [not_null]
  - name: stg_fred__observations
    description: FRED observations with series title, units and copyright flag.
    columns:
      - name: row_key
        tests: [unique, not_null]
  - name: stg_oecd__cli
    description: OECD composite leading indicators (amplitude adjusted), monthly by area.
    columns:
      - name: row_key
        tests: [unique, not_null]
  - name: stg_noaa__storm_tracks
    description: Atlantic storm positions and strength every 6 hours (HURDAT2), 1990 onward.
    columns:
      - name: row_key
        tests: [unique, not_null]
  - name: stg_noaa__active_storms
    description: Active storm snapshots, one row per storm per ingest run.
  - name: stg_gdelt__events
    description: Oil-relevant GDELT events with event date, news (publication) date and lag between them.
    columns:
      - name: event_id
        tests: [unique, not_null]
      - name: news_date
        tests: [not_null]
      - name: matched_by
        tests:
          - accepted_values:
              arguments:
                values: ['oil_word', 'conflict_place', 'diplomacy_place']
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "staging")

In [0]:
import json
for table in ["cftc_cot", "ice_cot"]:
    row = spark.sql(f"SELECT row_json FROM oil_shock.bronze.{table} ORDER BY report_date DESC LIMIT 1").collect()[0][0]
    keys = list(json.loads(row).keys())
    wanted = [k for k in keys if any(w in k.lower() for w in
              ["open_interest_all", "prod_merc", "swap", "m_money", "other_rept", "nonrept", "market", "date", "code"])]
    print(f"\n{table}: {len(keys)} fields. Relevant:")
    print(wanted[:40])

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/staging/stg_cot__positions.sql": """{% set n = "cast(regexp_replace(get_json_object(row_json, '$.{}'), ',', '') as double)" %}

with cftc as (
    select
        'CFTC'                                         as source,
        case market_code
            when '067651' then 'wti_nymex'
            when '111659' then 'rbob_nymex'
            when '022651' then 'ulsd_nymex'
        end                                            as contract_key,
        'futures_only'                                 as report_type,
        market_name,
        to_date(report_date)                           as report_date,
        {{ n.format('open_interest_all') }}            as open_interest,
        {{ n.format('prod_merc_positions_long') }}     as producer_long,
        {{ n.format('prod_merc_positions_short') }}    as producer_short,
        {{ n.format('swap_positions_long_all') }}      as swap_long,
        {{ n.format('swap__positions_short_all') }}    as swap_short,
        {{ n.format('m_money_positions_long_all') }}   as managed_money_long,
        {{ n.format('m_money_positions_short_all') }}  as managed_money_short,
        {{ n.format('other_rept_positions_long') }}    as other_reportable_long,
        {{ n.format('other_rept_positions_short') }}   as other_reportable_short,
        {{ n.format('nonrept_positions_long_all') }}   as nonreportable_long,
        {{ n.format('nonrept_positions_short_all') }}  as nonreportable_short,
        loaded_at
    from {{ source('bronze', 'cftc_cot') }}
),

ice as (
    select
        'ICE'                                          as source,
        case
            when market_name like 'ICE Brent Crude Futures and Options%' then 'brent_ice'
            when market_name like 'ICE Brent Crude Futures%'             then 'brent_ice'
            when market_name like 'ICE Gasoil Futures and Options%'      then 'gasoil_ice'
            when market_name like 'ICE Gasoil Futures%'                  then 'gasoil_ice'
            when market_name like 'ICE Dubai%'                           then 'dubai_ice'
        end                                            as contract_key,
        case when market_name like '%and Options%' then 'futures_and_options'
             else 'futures_only' end                   as report_type,
        market_name,
        to_date(report_date)                           as report_date,
        {{ n.format('Open_Interest_All') }}            as open_interest,
        {{ n.format('Prod_Merc_Positions_Long_All') }} as producer_long,
        {{ n.format('Prod_Merc_Positions_Short_All') }} as producer_short,
        {{ n.format('Swap_Positions_Long_All') }}      as swap_long,
        {{ n.format('Swap_Positions_Short_All') }}     as swap_short,
        {{ n.format('M_Money_Positions_Long_All') }}   as managed_money_long,
        {{ n.format('M_Money_Positions_Short_All') }}  as managed_money_short,
        {{ n.format('Other_Rept_Positions_Long_All') }} as other_reportable_long,
        {{ n.format('Other_Rept_Positions_Short_All') }} as other_reportable_short,
        {{ n.format('NonRept_Positions_Long_All') }}   as nonreportable_long,
        {{ n.format('NonRept_Positions_Short_All') }}  as nonreportable_short,
        loaded_at
    from {{ source('bronze', 'ice_cot') }}
),

unioned as (
    select * from cftc
    union all
    select * from ice
)

select
    concat_ws('|', contract_key, report_type, cast(report_date as string)) as row_key,
    *,
    managed_money_long - managed_money_short       as managed_money_net,
    producer_long - producer_short                 as producer_net,
    swap_long - swap_short                         as swap_net
from unioned
""",
"models/staging/stg_cot__positions.yml": """version: 2

models:
  - name: stg_cot__positions
    description: Weekly trader positions for oil futures (CFTC NYMEX WTI/RBOB/ULSD; ICE Brent, gasoil, Dubai), one row per contract, report type and week, with net positions.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: contract_key
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ['wti_nymex', 'rbob_nymex', 'ulsd_nymex', 'brent_ice', 'gasoil_ice', 'dubai_ice']
      - name: report_date
        tests: [not_null]
      - name: open_interest
        tests: [not_null]
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "stg_cot__positions")

display(spark.sql("""
    SELECT contract_key, report_type, MIN(report_date) AS first_week, MAX(report_date) AS last_week, COUNT(*) AS weeks,
           MAX_BY(managed_money_net, report_date) AS latest_money_manager_net
    FROM oil_shock.silver.stg_cot__positions
    GROUP BY contract_key, report_type ORDER BY contract_key, report_type
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

countries = """iso2,country_name,region,opec_member,opec_plus_partner,hormuz_exporter
AE,United Arab Emirates,Middle East,true,false,true
AL,Albania,Europe,false,false,false
AM,Armenia,Eurasia,false,false,false
AO,Angola,Africa,false,false,false
AR,Argentina,Central & South America,false,false,false
AT,Austria,Europe,false,false,false
AU,Australia,Asia & Oceania,false,false,false
AZ,Azerbaijan,Eurasia,false,true,false
BB,Barbados,Central & South America,false,false,false
BD,Bangladesh,Asia & Oceania,false,false,false
BE,Belgium,Europe,false,false,false
BG,Bulgaria,Europe,false,false,false
BH,Bahrain,Middle East,false,true,true
BM,Bermuda,North America,false,false,false
BN,Brunei,Asia & Oceania,false,true,false
BO,Bolivia,Central & South America,false,false,false
BR,Brazil,Central & South America,false,false,false
BY,Belarus,Eurasia,false,false,false
BZ,Belize,Central & South America,false,false,false
CA,Canada,North America,false,false,false
CH,Switzerland,Europe,false,false,false
CL,Chile,Central & South America,false,false,false
CN,China,Asia & Oceania,false,false,false
CO,Colombia,Central & South America,false,false,false
CR,Costa Rica,Central & South America,false,false,false
CU,Cuba,Central & South America,false,false,false
CY,Cyprus,Europe,false,false,false
CZ,Czechia,Europe,false,false,false
DE,Germany,Europe,false,false,false
DK,Denmark,Europe,false,false,false
DO,Dominican Republic,Central & South America,false,false,false
DZ,Algeria,Africa,true,false,false
EC,Ecuador,Central & South America,false,false,false
EE,Estonia,Europe,false,false,false
EG,Egypt,Africa,false,false,false
ES,Spain,Europe,false,false,false
FI,Finland,Europe,false,false,false
FR,France,Europe,false,false,false
GA,Gabon,Africa,true,false,false
GB,United Kingdom,Europe,false,false,false
GD,Grenada,Central & South America,false,false,false
GE,Georgia,Eurasia,false,false,false
GM,Gambia,Africa,false,false,false
GQ,Equatorial Guinea,Africa,true,false,false
GR,Greece,Europe,false,false,false
GT,Guatemala,Central & South America,false,false,false
GY,Guyana,Central & South America,false,false,false
HK,Hong Kong,Asia & Oceania,false,false,false
HN,Honduras,Central & South America,false,false,false
HR,Croatia,Europe,false,false,false
HT,Haiti,Central & South America,false,false,false
HU,Hungary,Europe,false,false,false
ID,Indonesia,Asia & Oceania,false,false,false
IE,Ireland,Europe,false,false,false
IN,India,Asia & Oceania,false,false,false
IQ,Iraq,Middle East,true,false,true
IR,Iran,Middle East,true,false,true
IS,Iceland,Europe,false,false,false
IT,Italy,Europe,false,false,false
JM,Jamaica,Central & South America,false,false,false
JP,Japan,Asia & Oceania,false,false,false
KR,South Korea,Asia & Oceania,false,false,false
KW,Kuwait,Middle East,true,false,true
KZ,Kazakhstan,Eurasia,false,true,false
LT,Lithuania,Europe,false,false,false
LU,Luxembourg,Europe,false,false,false
LV,Latvia,Europe,false,false,false
LY,Libya,Africa,true,false,false
MA,Morocco,Africa,false,false,false
MD,Moldova,Eurasia,false,false,false
MK,North Macedonia,Europe,false,false,false
MM,Myanmar,Asia & Oceania,false,false,false
MT,Malta,Europe,false,false,false
MU,Mauritius,Africa,false,false,false
MX,Mexico,North America,false,true,false
MY,Malaysia,Asia & Oceania,false,true,false
NE,Niger,Africa,false,false,false
NG,Nigeria,Africa,true,false,false
NI,Nicaragua,Central & South America,false,false,false
NL,Netherlands,Europe,false,false,false
NO,Norway,Europe,false,false,false
NP,Nepal,Asia & Oceania,false,false,false
NZ,New Zealand,Asia & Oceania,false,false,false
OM,Oman,Middle East,false,true,false
PA,Panama,Central & South America,false,false,false
PE,Peru,Central & South America,false,false,false
PG,Papua New Guinea,Asia & Oceania,false,false,false
PH,Philippines,Asia & Oceania,false,false,false
PL,Poland,Europe,false,false,false
PT,Portugal,Europe,false,false,false
PY,Paraguay,Central & South America,false,false,false
QA,Qatar,Middle East,false,false,true
RO,Romania,Europe,false,false,false
RS,Serbia,Europe,false,false,false
RU,Russia,Eurasia,false,true,false
SA,Saudi Arabia,Middle East,true,false,true
SD,Sudan,Africa,false,true,false
SE,Sweden,Europe,false,false,false
SG,Singapore,Asia & Oceania,false,false,false
SI,Slovenia,Europe,false,false,false
SK,Slovakia,Europe,false,false,false
SR,Suriname,Central & South America,false,false,false
SV,El Salvador,Central & South America,false,false,false
SY,Syria,Middle East,false,false,false
SZ,Eswatini,Africa,false,false,false
TH,Thailand,Asia & Oceania,false,false,false
TJ,Tajikistan,Eurasia,false,false,false
TN,Tunisia,Africa,false,false,false
TR,Turkiye,Europe,false,false,false
TT,Trinidad and Tobago,Central & South America,false,false,false
TW,Taiwan,Asia & Oceania,false,false,false
UA,Ukraine,Eurasia,false,false,false
US,United States,North America,false,false,false
UY,Uruguay,Central & South America,false,false,false
VE,Venezuela,Central & South America,true,false,false
VN,Vietnam,Asia & Oceania,false,false,false
YE,Yemen,Middle East,false,false,false
ZA,South Africa,Africa,false,false,false
"""

gdelt_regions = """fips_code,country_name,event_region
IR,Iran,Gulf
IZ,Iraq,Gulf
SA,Saudi Arabia,Gulf
KU,Kuwait,Gulf
AE,United Arab Emirates,Gulf
QA,Qatar,Gulf
MU,Oman,Gulf
BA,Bahrain,Gulf
YM,Yemen,Red Sea
DJ,Djibouti,Red Sea
ER,Eritrea,Red Sea
EG,Egypt,Red Sea
SU,Sudan,Red Sea
SO,Somalia,Red Sea
RS,Russia,Russia/Black Sea
UP,Ukraine,Russia/Black Sea
TU,Turkiye,Russia/Black Sea
KZ,Kazakhstan,Russia/Black Sea
AJ,Azerbaijan,Russia/Black Sea
LY,Libya,Africa
NI,Nigeria,Africa
AG,Algeria,Africa
AO,Angola,Africa
VE,Venezuela,Americas
US,United States,Americas
IS,Israel,Levant
"""

chokepoints = """chokepoint_name,route_region,oil_relevant
Strait of Hormuz,Middle East,true
Bab el-Mandeb Strait,Middle East,true
Suez Canal,Middle East,true
Cape of Good Hope,Africa,true
Malacca Strait,Asia,true
Bosporus Strait,Europe,true
Panama Canal,Americas,true
Gibraltar Strait,Europe,true
Dover Strait,Europe,true
Oresund Strait,Europe,true
Kerch Strait,Europe,true
Taiwan Strait,Asia,true
Korea Strait,Asia,true
Luzon Strait,Asia,true
Lombok Strait,Asia,true
Sunda Strait,Asia,true
Makassar Strait,Asia,true
Bohai Strait,Asia,false
Tsugaru Strait,Asia,false
Mindoro Strait,Asia,false
Balabac Strait,Asia,false
Ombai Strait,Asia,false
Torres Strait,Oceania,false
Yucatan Channel,Americas,true
Windward Passage,Americas,false
Mona Passage,Americas,false
Magellan Strait,Americas,false
Bering Strait,Arctic,false
"""

files = {
"dbt_project.yml": """name: oil_shock
version: "1.0.0"
config-version: 2
profile: oil_shock

model-paths: ["models"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]
seed-paths: ["seeds"]
test-paths: ["tests"]

models:
  oil_shock:
    staging:
      +schema: silver
      +materialized: view
    intermediate:
      +schema: silver
      +materialized: view
    marts:
      +schema: gold
      +materialized: table

seeds:
  oil_shock:
    +schema: silver
    ref_countries:
      +column_types:
        opec_member: boolean
        opec_plus_partner: boolean
        hormuz_exporter: boolean
    ref_chokepoints:
      +column_types:
        oil_relevant: boolean
""",
"seeds/ref_countries.csv": countries,
"seeds/ref_gdelt_regions.csv": gdelt_regions,
"seeds/ref_chokepoints.csv": chokepoints,
"models/marts/dim_date.sql": """with days as (
    select explode(sequence(to_date('2002-01-01'), to_date('2027-12-31'), interval 1 day)) as date_day
)

select
    date_day,
    year(date_day)                                   as year,
    quarter(date_day)                                as quarter,
    month(date_day)                                  as month,
    date_format(date_day, 'MMM')                     as month_name,
    date_trunc('MONTH', date_day)                    as month_start,
    date_trunc('WEEK', date_day)                     as week_start,
    dayofweek(date_day)                              as day_of_week,
    dayofweek(date_day) in (1, 7)                    as is_weekend,
    month(date_day) between 6 and 8                  as is_driving_season,
    month(date_day) in (10, 11, 12, 1, 2, 3)         as is_heating_season,
    month(date_day) between 6 and 11                 as is_hurricane_season,
    date_day >= to_date('2026-02-28')                as is_war_period
from days
""",
"models/marts/dim_country.sql": """select
    iso2                                             as country_code,
    country_name,
    region,
    opec_member,
    opec_plus_partner,
    opec_member or opec_plus_partner                 as opec_plus,
    hormuz_exporter
from {{ ref('ref_countries') }}
""",
"models/marts/dim_chokepoint.sql": """with ids as (
    select distinct chokepoint_id, chokepoint_name
    from {{ ref('stg_portwatch__chokepoints') }}
)

select
    i.chokepoint_id,
    i.chokepoint_name,
    coalesce(r.route_region, 'Other')                as route_region,
    coalesce(r.oil_relevant, false)                  as oil_relevant
from ids i
left join {{ ref('ref_chokepoints') }} r
    on i.chokepoint_name = r.chokepoint_name
""",
"models/marts/dim_event_region.sql": """select distinct event_region
from {{ ref('ref_gdelt_regions') }}
union
select 'Other' as event_region
""",
"models/marts/_marts_dims.yml": """version: 2

models:
  - name: dim_date
    description: Calendar 2002-2027 with season flags (driving, heating, hurricane) and war period (from 2026-02-28).
    columns:
      - name: date_day
        tests: [unique, not_null]
  - name: dim_country
    description: JODI countries with region, OPEC and OPEC+ membership (2026) and Hormuz exporter flag.
    columns:
      - name: country_code
        tests: [unique, not_null]
      - name: region
        tests: [not_null]
  - name: dim_chokepoint
    description: PortWatch chokepoints with route region and oil relevance.
    columns:
      - name: chokepoint_id
        tests: [unique, not_null]
      - name: route_region
        tests:
          - not_null
  - name: dim_event_region
    description: Event regions used to group GDELT events.
    columns:
      - name: event_region
        tests: [unique, not_null]
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("seed")
run_dbt("build", "--select", "dim_date dim_country dim_chokepoint dim_event_region")

display(spark.sql("""
    SELECT 'JODI countries without a match' AS check, COUNT(*) AS n
    FROM (SELECT DISTINCT country_code FROM oil_shock.silver.stg_jodi__oil) j
    LEFT JOIN oil_shock.gold.dim_country c USING (country_code) WHERE c.country_code IS NULL
    UNION ALL
    SELECT 'Chokepoints without a region', COUNT(*) FROM oil_shock.gold.dim_chokepoint WHERE route_region = 'Other'
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

price_series = """series_id,price_name,price_group,market,unit_original,to_usd_per_bbl
RBRTE,Brent physical,crude,North Sea (Europe),USD/bbl,1
RWTC,WTI spot,crude,Cushing (US),USD/bbl,1
EER_EPMRU_PF4_Y35NY_DPG,Gasoline,product,New York Harbor (US),USD/gal,42
EER_EPD2DXL0_PF4_Y35NY_DPG,Ultra-low sulfur diesel,product,New York Harbor (US),USD/gal,42
EER_EPJK_PF4_RGC_DPG,Jet fuel,product,US Gulf Coast,USD/gal,42
EER_EPD2F_PF4_Y35NY_DPG,Heating oil,product,New York Harbor (US),USD/gal,42
RCLC1,WTI futures contract 1 (to Apr 2024),crude_futures,NYMEX (US),USD/bbl,1
RCLC2,WTI futures contract 2 (to Apr 2024),crude_futures,NYMEX (US),USD/bbl,1
RCLC3,WTI futures contract 3 (to Apr 2024),crude_futures,NYMEX (US),USD/bbl,1
RCLC4,WTI futures contract 4 (to Apr 2024),crude_futures,NYMEX (US),USD/bbl,1
RNGWHHD,Henry Hub natural gas,gas,Henry Hub (US),USD/MMBtu,
POILDUBUSDM,Dubai (IMF monthly average),crude,Middle East,USD/bbl,1
POILBREUSDM,Brent (IMF monthly average),crude,North Sea (Europe),USD/bbl,1
POILWTIUSDM,WTI (IMF monthly average),crude,US,USD/bbl,1
"""

files = {
"seeds/ref_price_series.csv": price_series,
"models/marts/dim_price_series.sql": """select
    series_id,
    price_name,
    price_group,
    market,
    unit_original,
    case when to_usd_per_bbl is null then unit_original else 'USD/bbl' end as unit_reported,
    to_usd_per_bbl
from {{ ref('ref_price_series') }}
""",
"models/marts/fct_price_daily.sql": """select
    concat_ws('|', e.series_id, cast(e.period_date as string)) as row_key,
    e.period_date                                           as price_date,
    e.series_id,
    p.price_name,
    p.price_group,
    e.value                                                 as price_original,
    p.unit_original,
    case when p.to_usd_per_bbl is null then e.value
         else e.value * p.to_usd_per_bbl end                as price,
    case when p.to_usd_per_bbl is null then p.unit_original
         else 'USD/bbl' end                                 as price_unit
from {{ ref('stg_eia__series') }} e
join {{ ref('ref_price_series') }} p
    on e.series_id = p.series_id
where e.frequency = 'daily'
  and e.data_kind = 'actual'
""",
"models/marts/fct_spread_daily.sql": """with wide as (
    select
        price_date,
        max(case when series_id = 'RBRTE' then price end)                      as brent,
        max(case when series_id = 'RWTC' then price end)                       as wti,
        max(case when series_id = 'EER_EPMRU_PF4_Y35NY_DPG' then price end)    as gasoline,
        max(case when series_id = 'EER_EPD2DXL0_PF4_Y35NY_DPG' then price end) as diesel,
        max(case when series_id = 'EER_EPJK_PF4_RGC_DPG' then price end)       as jet
    from {{ ref('fct_price_daily') }}
    group by price_date
)

select
    price_date,
    brent,
    wti,
    brent - wti                 as brent_wti_spread,
    gasoline - brent            as gasoline_crack,
    diesel - brent              as diesel_crack,
    jet - brent                 as jet_crack
from wide
where brent is not null
""",
"models/marts/fct_price_monthly.sql": """select
    concat_ws('|', f.series_id, cast(f.obs_date as string)) as row_key,
    f.obs_date                                              as month_start,
    f.series_id,
    p.price_name,
    f.value                                                 as price,
    'USD/bbl'                                               as price_unit,
    'Source: International Monetary Fund, via FRED'         as attribution
from {{ ref('stg_fred__observations') }} f
join {{ ref('ref_price_series') }} p
    on f.series_id = p.series_id
""",
"models/marts/_marts_prices.yml": """version: 2

models:
  - name: dim_price_series
    description: Price series with name, group, market and units (products converted to USD/bbl).
    columns:
      - name: series_id
        tests: [unique, not_null]
  - name: fct_price_daily
    description: Daily prices from EIA. Brent is physical spot (not futures); WTI futures end April 2024. Products converted from USD/gal to USD/bbl.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: price
        tests: [not_null]
  - name: fct_spread_daily
    description: Brent minus WTI, and product cracks (product minus Brent physical), USD/bbl.
    columns:
      - name: price_date
        tests: [unique, not_null]
  - name: fct_price_monthly
    description: IMF monthly average prices for Dubai, Brent and WTI (attribution required).
    columns:
      - name: row_key
        tests: [unique, not_null]
""",
"tests/assert_oil_prices_positive.sql": """-- Fails if any oil price is zero or negative.
-- WTI fell below zero on 2020-04-20 (spot and futures); those two days are allowed.
select *
from {{ ref('fct_price_daily') }}
where price_group in ('crude', 'product', 'crude_futures')
  and price <= 0
  and price_date <> to_date('2020-04-20')
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("seed", "--select", "ref_price_series")
run_dbt("build", "--select", "dim_price_series fct_price_daily fct_spread_daily fct_price_monthly assert_oil_prices_positive")

display(spark.sql("""
    SELECT price_date, ROUND(brent, 2) AS brent, ROUND(wti, 2) AS wti,
           ROUND(brent_wti_spread, 2) AS brent_wti, ROUND(gasoline_crack, 2) AS gasoline_crack,
           ROUND(diesel_crack, 2) AS diesel_crack, ROUND(jet_crack, 2) AS jet_crack
    FROM oil_shock.gold.fct_spread_daily
    ORDER BY price_date DESC LIMIT 5
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/marts/fct_us_weekly_balance.sql": """with w as (
    select
        period_date as week_ending,
        max(case when series_id = 'WCESTUS1' then value end) as crude_stocks_commercial_kbbl,
        max(case when series_id = 'WCSSTUS1' then value end) as crude_stocks_spr_kbbl,
        max(case when series_id = 'WGTSTUS1' then value end) as gasoline_stocks_kbbl,
        max(case when series_id = 'WDISTUS1' then value end) as distillate_stocks_kbbl,
        max(case when series_id = 'WCRFPUS2' then value end) as crude_production_kbd,
        max(case when series_id = 'WPULEUS3' then value end) as refinery_utilization_pct,
        max(case when series_id = 'WRPUPUS2' then value end) as product_supplied_kbd,
        max(case when series_id = 'WCRIMUS2' then value end) as crude_imports_kbd,
        max(case when series_id = 'WCREXUS2' then value end) as crude_exports_kbd
    from {{ ref('stg_eia__series') }}
    where frequency = 'weekly' and data_kind = 'actual'
    group by period_date
)

select
    *,
    crude_stocks_commercial_kbbl
      - lag(crude_stocks_commercial_kbbl) over (order by week_ending) as crude_stocks_change_kbbl,
    crude_stocks_spr_kbbl
      - lag(crude_stocks_spr_kbbl) over (order by week_ending)        as spr_change_kbbl,
    crude_imports_kbd - crude_exports_kbd                              as crude_net_imports_kbd
from w
""",
"models/marts/_marts_us_weekly.yml": """version: 2

models:
  - name: fct_us_weekly_balance
    description: US weekly oil balance from EIA (stocks in thousand barrels, flows in thousand barrels per day), with weekly stock changes. US-only data, used as the best free high-frequency signal.
    columns:
      - name: week_ending
        tests: [unique, not_null]
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "fct_us_weekly_balance")

display(spark.sql("""
    SELECT week_ending, crude_stocks_commercial_kbbl, crude_stocks_change_kbbl, crude_stocks_spr_kbbl,
           crude_production_kbd, refinery_utilization_pct, product_supplied_kbd, crude_net_imports_kbd
    FROM oil_shock.gold.fct_us_weekly_balance ORDER BY week_ending DESC LIMIT 5
"""))

display(spark.sql("""
    SELECT category, energy_product, flow_breakdown, unit_measure,
           SUM(CASE WHEN is_reported THEN 1 ELSE 0 END) AS reported_rows
    FROM oil_shock.silver.stg_jodi__oil
    WHERE period_date >= '2025-01-01'
    GROUP BY ALL
    HAVING reported_rows > 0
    ORDER BY category, energy_product, flow_breakdown, unit_measure
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

products = """product_code,product_name,product_group
CRUDEOIL,Crude oil,crude
NGL,Natural gas liquids,crude
OTHERCRUDE,Other primary (feedstocks and additives),crude
TOTCRUDE,Total primary (crude + NGL + other),crude_total
LPG,LPG,product
NAPHTHA,Naphtha,product
GASOLINE,Gasoline,product
KEROSENE,Kerosene,product
JETKERO,Jet fuel (kerosene type),product
GASDIES,Diesel and gasoil,product
RESFUEL,Fuel oil,product
ONONSPEC,Other products,product
TOTPRODS,Total products,product_total
"""

flows = """flow_code,flow_name,measure
INDPROD,Production,flow
OSOURCES,Production from other sources,flow
TOTIMPSB,Imports,flow
TOTEXPSB,Exports,flow
TRANSBAK,Products transferred back,flow
DIRECUSE,Direct use,flow
STATDIFF,Statistical difference,flow
REFINOBS,Refinery intake,flow
REFGROUT,Refinery output,flow
RECEIPTS,Receipts,flow
PTRANSF,Products transferred,flow
IPTRANSF,Interproduct transfers,flow
TOTDEMO,Demand,flow
STOCKCH,Stock change,stock
CLOSTLV,Closing stocks,stock
"""

files = {
"seeds/ref_jodi_products.csv": products,
"seeds/ref_jodi_flows.csv": flows,
"models/marts/fct_oil_balance_monthly.sql": """select
    j.row_key,
    j.country_code,
    j.period_date,
    j.energy_product                       as product_code,
    p.product_name,
    p.product_group,
    j.flow_breakdown                       as flow_code,
    f.flow_name,
    f.measure,
    j.unit_measure,
    j.obs_value                            as value,
    j.assessment_code
from {{ ref('stg_jodi__oil') }} j
join {{ ref('ref_jodi_products') }} p on j.energy_product = p.product_code
join {{ ref('ref_jodi_flows') }} f    on j.flow_breakdown = f.flow_code
where j.is_reported
  and ((f.measure = 'flow'  and j.unit_measure = 'KBD')
    or (f.measure = 'stock' and j.unit_measure = 'KBBL'))
""",
"models/marts/fct_country_balance_monthly.sql": """with b as (
    select * from {{ ref('fct_oil_balance_monthly') }}
)

select
    concat_ws('|', country_code, cast(period_date as string)) as row_key,
    country_code,
    period_date,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'INDPROD'  then value end) as crude_production_kbd,
    max(case when product_code = 'NGL'      and flow_code = 'INDPROD'  then value end) as ngl_production_kbd,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'TOTIMPSB' then value end) as crude_imports_kbd,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'TOTEXPSB' then value end) as crude_exports_kbd,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'REFINOBS' then value end) as crude_refinery_intake_kbd,
    max(case when product_code = 'TOTPRODS' and flow_code = 'TOTDEMO'  then value end) as product_demand_kbd,
    max(case when product_code = 'TOTPRODS' and flow_code = 'REFGROUT' then value end) as product_output_kbd,
    max(case when product_code = 'TOTPRODS' and flow_code = 'TOTIMPSB' then value end) as product_imports_kbd,
    max(case when product_code = 'TOTPRODS' and flow_code = 'TOTEXPSB' then value end) as product_exports_kbd,
    max(case when product_code = 'GASDIES'  and flow_code = 'TOTDEMO'  then value end) as diesel_demand_kbd,
    max(case when product_code = 'JETKERO'  and flow_code = 'TOTDEMO'  then value end) as jet_demand_kbd,
    max(case when product_code = 'GASOLINE' and flow_code = 'TOTDEMO'  then value end) as gasoline_demand_kbd,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'CLOSTLV'  then value end) as crude_stocks_kbbl,
    max(case when product_code = 'TOTPRODS' and flow_code = 'CLOSTLV'  then value end) as product_stocks_kbbl
from b
group by country_code, period_date
""",
"models/marts/_marts_jodi.yml": """version: 2

models:
  - name: fct_oil_balance_monthly
    description: JODI monthly oil balances by country, product and flow. Flows in thousand barrels per day (KBD), stocks in thousand barrels (KBBL). Reported values only.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: country_code
        tests:
          - not_null
          - relationships:
              arguments:
                to: ref('dim_country')
                field: country_code
  - name: fct_country_balance_monthly
    description: One row per country and month with the main oil balance items (KBD flows, KBBL stocks).
    columns:
      - name: row_key
        tests: [unique, not_null]
""",
"tests/assert_eia_brent_matches_imf.sql": """{{ config(severity = 'warn') }}
-- Warns if EIA's monthly average Brent physical differs from the IMF monthly Brent by more than 5% (from 2019).
with eia as (
    select date_trunc('MONTH', price_date) as month_start, avg(price) as eia_brent
    from {{ ref('fct_price_daily') }}
    where series_id = 'RBRTE'
    group by 1
),
imf as (
    select month_start, price as imf_brent
    from {{ ref('fct_price_monthly') }}
    where series_id = 'POILBREUSDM'
)
select
    i.month_start,
    round(e.eia_brent, 2)                                   as eia_brent,
    round(i.imf_brent, 2)                                   as imf_brent,
    round(100 * (e.eia_brent - i.imf_brent) / i.imf_brent, 1) as pct_diff
from imf i
join eia e on cast(i.month_start as date) = cast(e.month_start as date)
where i.month_start >= '2019-01-01'
  and abs(e.eia_brent - i.imf_brent) / i.imf_brent > 0.05
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("seed", "--select", "ref_jodi_products ref_jodi_flows")
run_dbt("build", "--select", "fct_oil_balance_monthly fct_country_balance_monthly assert_eia_brent_matches_imf")

display(spark.sql("""
    SELECT b.country_code, c.country_name, b.period_date, b.crude_production_kbd, b.crude_exports_kbd,
           b.product_demand_kbd, b.crude_stocks_kbbl
    FROM oil_shock.gold.fct_country_balance_monthly b
    JOIN oil_shock.gold.dim_country c USING (country_code)
    WHERE b.country_code IN ('SA', 'IQ', 'AE', 'KW', 'US', 'RU')
      AND b.period_date >= '2026-01-01'
    ORDER BY b.country_code, b.period_date
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/marts/fct_chokepoint_daily.sql": """with base as (
    select
        s.chokepoint_id,
        s.chokepoint_name,
        s.transit_date,
        s.ships_total,
        s.ships_tanker,
        s.capacity_tanker_mt,
        avg(s.ships_total)  over w7 as ships_total_7d_avg,
        avg(s.ships_tanker) over w7 as ships_tanker_7d_avg
    from {{ ref('stg_portwatch__chokepoints') }} s
    window w7 as (partition by s.chokepoint_id order by s.transit_date
                  rows between 6 preceding and current row)
),

normal_2025 as (
    select
        chokepoint_id,
        avg(ships_total)  as ships_total_2025_avg,
        avg(ships_tanker) as ships_tanker_2025_avg
    from {{ ref('stg_portwatch__chokepoints') }}
    where year(transit_date) = 2025
    group by chokepoint_id
)

select
    concat_ws('|', b.chokepoint_id, cast(b.transit_date as string)) as row_key,
    b.*,
    n.ships_total_2025_avg,
    n.ships_tanker_2025_avg,
    round(100 * b.ships_total_7d_avg  / nullif(n.ships_total_2025_avg, 0), 1)  as ships_total_pct_of_2025,
    round(100 * b.ships_tanker_7d_avg / nullif(n.ships_tanker_2025_avg, 0), 1) as ships_tanker_pct_of_2025
from base b
left join normal_2025 n using (chokepoint_id)
""",
"models/marts/_marts_chokepoints.yml": """version: 2

models:
  - name: fct_chokepoint_daily
    description: Daily ship transits per chokepoint with 7-day averages and percent of the 2025 (pre-war) daily average. Sources UN Global Platform; IMF PortWatch.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: chokepoint_id
        tests:
          - relationships:
              arguments:
                to: ref('dim_chokepoint')
                field: chokepoint_id
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "fct_chokepoint_daily")

display(spark.sql("""
    SELECT chokepoint_name, transit_date, ships_total, ships_tanker,
           ROUND(ships_total_7d_avg, 1) AS total_7d, ships_total_pct_of_2025 AS pct_of_normal,
           ships_tanker_pct_of_2025 AS tanker_pct_of_normal
    FROM oil_shock.gold.fct_chokepoint_daily
    WHERE chokepoint_name IN ('Strait of Hormuz', 'Bab el-Mandeb Strait', 'Suez Canal', 'Cape of Good Hope', 'Malacca Strait')
      AND transit_date = (SELECT MAX(transit_date) FROM oil_shock.gold.fct_chokepoint_daily)
    ORDER BY pct_of_normal
"""))

display(spark.sql("""
    WITH eia AS (
        SELECT date_trunc('MONTH', price_date) AS month_start, AVG(price) AS eia_brent
        FROM oil_shock.gold.fct_price_daily WHERE series_id = 'RBRTE' GROUP BY 1
    ),
    imf AS (
        SELECT month_start, price AS imf_brent FROM oil_shock.gold.fct_price_monthly WHERE series_id = 'POILBREUSDM'
    )
    SELECT CAST(i.month_start AS DATE) AS month, ROUND(e.eia_brent, 2) AS eia_brent_physical,
           ROUND(i.imf_brent, 2) AS imf_brent_avg, ROUND(100 * (e.eia_brent - i.imf_brent) / i.imf_brent, 1) AS pct_diff
    FROM imf i JOIN eia e ON CAST(i.month_start AS DATE) = CAST(e.month_start AS DATE)
    WHERE i.month_start >= '2025-10-01'
    ORDER BY month
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/marts/fct_positioning_weekly.sql": """with p as (
    select * from {{ ref('stg_cot__positions') }}
),

calc as (
    select
        row_key,
        source,
        contract_key,
        report_type,
        report_date,
        open_interest,
        managed_money_long,
        managed_money_short,
        managed_money_net,
        producer_net,
        swap_net,
        managed_money_net - lag(managed_money_net) over w      as managed_money_net_change,
        100 * managed_money_net / nullif(open_interest, 0)      as managed_money_net_pct_oi,
        avg(managed_money_net)    over w52                      as mm_net_52w_avg,
        stddev(managed_money_net) over w52                      as mm_net_52w_std
    from p
    window
        w   as (partition by contract_key, report_type order by report_date),
        w52 as (partition by contract_key, report_type order by report_date
                rows between 51 preceding and current row)
)

select
    *,
    round((managed_money_net - mm_net_52w_avg) / nullif(mm_net_52w_std, 0), 2) as crowding_zscore
from calc
""",
"models/marts/fct_macro_daily.sql": """with d as (
    select
        obs_date,
        max(case when series_id = 'DTWEXBGS' then value end) as usd_index,
        max(case when series_id = 'DFF'      then value end) as fed_funds_rate,
        max(case when series_id = 'DGS10'    then value end) as treasury_10y_yield,
        max(case when series_id = 'VIXCLS'   then value end) as vix_model_only
    from {{ ref('stg_fred__observations') }}
    where series_id in ('DTWEXBGS', 'DFF', 'DGS10', 'VIXCLS')
      and obs_date >= '2000-01-01'
    group by obs_date
)

select
    *,
    case
        when vix_model_only is null then null
        when vix_model_only < 15 then 'low'
        when vix_model_only <= 25 then 'normal'
        else 'high'
    end as fear_level
from d
""",
"models/marts/fct_macro_monthly.sql": """with m as (
    select
        obs_date as month_start,
        max(case when series_id = 'CPIAUCSL' then value end) as us_cpi_index,
        max(case when series_id = 'INDPRO'   then value end) as us_industrial_production_index
    from {{ ref('stg_fred__observations') }}
    where series_id in ('CPIAUCSL', 'INDPRO')
    group by obs_date
)

select
    *,
    round(100 * (us_cpi_index / lag(us_cpi_index, 12) over (order by month_start) - 1), 2)
        as us_inflation_yoy_pct,
    round(100 * (us_industrial_production_index
                 / lag(us_industrial_production_index, 12) over (order by month_start) - 1), 2)
        as us_industrial_production_yoy_pct
from m
""",
"models/marts/fct_leading_indicator_monthly.sql": """select
    row_key,
    area_code,
    area_name,
    period_date,
    cli_value,
    cli_value - lag(cli_value, 3) over (partition by area_code order by period_date) as cli_change_3m,
    case
        when cli_value - lag(cli_value, 3) over (partition by area_code order by period_date) > 0 then 'rising'
        when cli_value - lag(cli_value, 3) over (partition by area_code order by period_date) < 0 then 'falling'
        else 'flat'
    end as cli_direction
from {{ ref('stg_oecd__cli') }}
""",
"models/marts/_marts_drivers.yml": """version: 2

models:
  - name: fct_positioning_weekly
    description: Weekly trader positions with money-manager net, weekly change, share of open interest and a 52-week crowding z-score (CFTC NYMEX; ICE Brent, gasoil, Dubai).
    columns:
      - name: row_key
        tests: [unique, not_null]
  - name: fct_macro_daily
    description: Daily dollar index, Fed funds rate, 10-year yield and VIX (VIX for models only; report shows fear_level, credited to Cboe via FRED).
    columns:
      - name: obs_date
        tests: [unique, not_null]
      - name: fear_level
        tests:
          - accepted_values:
              arguments:
                values: ['low', 'normal', 'high']
  - name: fct_macro_monthly
    description: US CPI and industrial production with year-on-year changes.
    columns:
      - name: month_start
        tests: [unique, not_null]
  - name: fct_leading_indicator_monthly
    description: OECD composite leading indicators by economy with 3-month change and direction.
    columns:
      - name: row_key
        tests: [unique, not_null]
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "fct_positioning_weekly fct_macro_daily fct_macro_monthly fct_leading_indicator_monthly")

display(spark.sql("""
    SELECT contract_key, report_type, report_date, managed_money_net, managed_money_net_change,
           ROUND(managed_money_net_pct_oi, 1) AS net_pct_oi, crowding_zscore
    FROM oil_shock.gold.fct_positioning_weekly
    WHERE report_date = (SELECT MAX(report_date) FROM oil_shock.gold.fct_positioning_weekly)
    ORDER BY contract_key, report_type
"""))

display(spark.sql("""
    SELECT area_code, area_name, period_date, ROUND(cli_value, 2) AS cli, ROUND(cli_change_3m, 2) AS change_3m, cli_direction
    FROM oil_shock.gold.fct_leading_indicator_monthly
    WHERE period_date = (SELECT MAX(period_date) FROM oil_shock.gold.fct_leading_indicator_monthly)
    ORDER BY cli DESC
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/marts/fct_steo_forecast.sql": """select
    row_key,
    steo_vintage                                            as release_month,
    series_id,
    series_label,
    units,
    period_date                                             as target_month,
    value,
    period_date >= to_date(concat(steo_vintage, '-01'))     as is_forecast,
    months_between(period_date, to_date(concat(steo_vintage, '-01'))) as months_ahead
from {{ ref('stg_eia__series') }}
where data_kind = 'forecast'
""",
"models/marts/fct_storm_track.sql": """with history as (
    select
        storm_id,
        storm_name,
        obs_ts,
        status,
        latitude,
        longitude,
        max_wind_kt,
        'hurdat2' as source
    from {{ ref('stg_noaa__storm_tracks') }}
),

live as (
    select
        upper(storm_id)     as storm_id,
        upper(storm_name)   as storm_name,
        snapshot_at         as obs_ts,
        classification      as status,
        latitude,
        longitude,
        intensity_kt        as max_wind_kt,
        'active_snapshot'   as source
    from {{ ref('stg_noaa__active_storms') }}
    where snapshot_at > (select max(obs_ts) from history)
)

select
    *,
    latitude between 18 and 31 and longitude between -98 and -80 as in_gulf_of_mexico
from (select * from history union all select * from live)
""",
"models/marts/fct_gulf_storm_daily.sql": """select
    cast(obs_ts as date)              as storm_date,
    count(distinct storm_id)          as storms_in_gulf,
    max(max_wind_kt)                  as max_wind_kt_in_gulf,
    max(case when status = 'HU' then 1 else 0 end) = 1 as hurricane_in_gulf
from {{ ref('fct_storm_track') }}
where in_gulf_of_mexico
group by cast(obs_ts as date)
""",
"models/marts/_marts_forecast_storms.yml": """version: 2

models:
  - name: fct_steo_forecast
    description: EIA Short-Term Energy Outlook values by release month (vintage); is_forecast marks months not yet past at release.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: release_month
        tests: [not_null]
  - name: fct_storm_track
    description: Atlantic storm positions (HURDAT2 history plus live snapshots) with a Gulf of Mexico flag (lat 18-31N, lon 98-80W).
  - name: fct_gulf_storm_daily
    description: Daily count of storms in the Gulf of Mexico and the strongest wind; weather driver for US offshore production.
    columns:
      - name: storm_date
        tests: [unique, not_null]
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "fct_steo_forecast fct_storm_track fct_gulf_storm_daily")

display(spark.sql("""
    SELECT release_month, target_month, ROUND(value, 1) AS brent_forecast, months_ahead
    FROM oil_shock.gold.fct_steo_forecast
    WHERE series_id = 'BREPUUS' AND is_forecast
    ORDER BY target_month
"""))

display(spark.sql("""
    SELECT * FROM oil_shock.gold.fct_gulf_storm_daily
    WHERE storm_date >= '2024-06-01' ORDER BY storm_date DESC LIMIT 10
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/staging/stg_noaa__bdeck.sql": """select
    concat_ws('|', storm_id, obs_time)     as row_key,
    storm_id,
    name                                   as storm_name,
    to_timestamp(obs_time)                 as obs_ts,
    year(to_timestamp(obs_time))           as season,
    status,
    latitude,
    longitude,
    max_wind_kt,
    min_pressure_mb,
    source_file,
    loaded_at
from {{ source('bronze', 'noaa_bdeck') }}
""",
"models/marts/fct_storm_track.sql": """with history as (
    select storm_id, storm_name, obs_ts, status, latitude, longitude, max_wind_kt,
           'hurdat2' as source
    from {{ ref('stg_noaa__storm_tracks') }}
),

season as (
    select storm_id, storm_name, obs_ts, status, latitude, longitude, max_wind_kt,
           'bdeck' as source
    from {{ ref('stg_noaa__bdeck') }}
    where season > (select max(season) from {{ ref('stg_noaa__storm_tracks') }})
),

live as (
    select upper(storm_id) as storm_id, upper(storm_name) as storm_name, snapshot_at as obs_ts,
           classification as status, latitude, longitude, intensity_kt as max_wind_kt,
           'active_snapshot' as source
    from {{ ref('stg_noaa__active_storms') }}
    where snapshot_at > (select coalesce(max(obs_ts), timestamp '1900-01-01') from season)
)

select
    *,
    latitude between 18 and 31 and longitude between -98 and -80 as in_gulf_of_mexico
from (
    select * from history
    union all select * from season
    union all select * from live
)
""",
}

for path, content in files.items():
    with open(f"{DBT}/{path}", "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

# Add the new bronze table to the sources list
src_path = f"{DBT}/models/staging/_sources.yml"
src = open(src_path).read()
if "noaa_bdeck" not in src:
    src = src.replace(
        "      - name: gdelt_events",
        "      - name: noaa_bdeck\n        description: NOAA NHC current-season Atlantic best tracks (b-decks), refreshed each run.\n      - name: gdelt_events")
    open(src_path, "w").write(src)
    print("Updated: dbt/models/staging/_sources.yml")

run_dbt("build", "--select", "stg_noaa__bdeck fct_storm_track fct_gulf_storm_daily")

display(spark.sql("""
    SELECT * FROM oil_shock.gold.fct_gulf_storm_daily
    WHERE storm_date >= '2026-06-01' ORDER BY storm_date DESC LIMIT 12
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"models/intermediate/int_gdelt__events_clean.sql": """select
    e.event_id,
    e.event_date,
    e.news_date,
    e.news_ts,
    e.days_event_before_news,
    e.actor1_name,
    e.actor2_name,
    e.event_code,
    e.event_base_code,
    e.event_root_code,
    e.goldstein,
    e.num_mentions,
    e.num_sources,
    e.avg_tone,
    e.geo_country,
    e.geo_name,
    coalesce(r.event_region, 'Other') as event_region,
    e.matched_by,
    e.source_url
from {{ ref('stg_gdelt__events') }} e
left join {{ ref('ref_gdelt_regions') }} r
    on e.geo_country = r.fips_code
where e.days_event_before_news between 0 and 7
""",
"models/intermediate/int_gdelt__stories.sql": """{{ config(
    materialized = 'incremental',
    unique_key = 'story_id',
    incremental_strategy = 'merge'
) }}

with recent_urls as (
    select distinct source_url
    from {{ ref('int_gdelt__events_clean') }}
    {% if is_incremental() %}
    where news_date >= (select date_sub(max(news_date), 3) from {{ this }})
    {% endif %}
),

ev as (
    select e.*
    from {{ ref('int_gdelt__events_clean') }} e
    join recent_urls u on e.source_url = u.source_url
),

ranked as (
    select
        *,
        row_number() over (partition by source_url order by num_mentions desc, event_id) as rn
    from ev
),

agg as (
    select
        source_url,
        min(news_date)                          as news_date,
        min(news_ts)                            as first_seen_ts,
        count(*)                                as num_events,
        max(num_mentions)                       as top_event_mentions,
        sum(num_mentions)                       as total_mentions,
        max(num_sources)                        as max_sources,
        min(goldstein)                          as min_goldstein,
        avg(avg_tone)                           as avg_tone,
        max(case when matched_by = 'oil_word' then 1 else 0 end) = 1 as has_oil_word
    from ev
    group by source_url
),

top_event as (
    select source_url, event_region, event_root_code, event_base_code, event_code,
           actor1_name, actor2_name, geo_name
    from ranked
    where rn = 1
),

joined as (
    select
        md5(a.source_url) as story_id,
        a.*,
        t.event_region,
        t.event_root_code,
        t.event_base_code,
        t.event_code,
        t.actor1_name,
        t.actor2_name,
        t.geo_name,
        lower(regexp_replace(
            regexp_replace(regexp_extract(a.source_url, '([^/]+)/?$', 1), '\\\\.(html?|php|aspx?)$', ''),
            '[-_]+', ' ')) as headline_slug,
        lower(a.source_url) as url_lc
    from agg a
    join top_event t on a.source_url = t.source_url
)

select
    story_id,
    source_url,
    headline_slug,
    news_date,
    first_seen_ts,
    event_region,
    event_root_code,
    event_base_code,
    event_code,
    actor1_name,
    actor2_name,
    geo_name,
    num_events,
    top_event_mentions,
    total_mentions,
    max_sources,
    min_goldstein,
    avg_tone,
    has_oil_word,
    case
        when url_lc rlike 'tanker|vessel|ship|cargo' and event_root_code in ('17', '18', '19', '20')
            then 'vessel_attack_or_seizure'
        when url_lc rlike 'refiner|pipeline|terminal|oilfield|oil-field|facility|depot|plant'
             and event_root_code in ('18', '19', '20')
            then 'infrastructure_strike'
        when url_lc rlike 'hormuz|strait|red-sea|bab-el|suez|chokepoint|blockade'
            then 'chokepoint'
        when event_base_code = '163' or url_lc rlike 'sanction|embargo'
            then 'sanctions'
        when url_lc rlike 'opec'
            then 'opec_plus'
        when event_root_code in ('03', '04', '05', '06', '08') or url_lc rlike 'ceasefire|truce|talks|negotiat'
            then 'talks_or_ceasefire'
        else 'other'
    end as category_rule,
    round(
        case event_root_code
            when '20' then 5 when '19' then 4 when '18' then 4 when '17' then 3
            when '15' then 2 when '16' then 2 when '13' then 1.5 when '14' then 1
            else 0.5
        end
        * ln(1 + total_mentions)
        * case when has_oil_word then 1.5 else 1 end
    , 2) as severity_score
from joined
""",
"models/marts/fct_risk_index_daily.sql": """with stories as (
    select news_date, event_region, severity_score from {{ ref('int_gdelt__stories') }}
    union all
    select news_date, 'Global' as event_region, severity_score from {{ ref('int_gdelt__stories') }}
),

daily as (
    select
        news_date,
        event_region,
        count(*)            as stories,
        sum(severity_score) as severity_total
    from stories
    group by news_date, event_region
),

volume as (
    select news_date, count(*) as gdelt_events
    from {{ ref('stg_gdelt__events') }}
    group by news_date
),

volume_flag as (
    select
        news_date,
        gdelt_events,
        gdelt_events < 0.5 * percentile_approx(gdelt_events, 0.5)
            over (order by news_date rows between 30 preceding and 1 preceding) as is_data_gap
    from volume
),

scored as (
    select
        d.*,
        avg(severity_total)    over w90 as severity_90d_avg,
        stddev(severity_total) over w90 as severity_90d_std
    from daily d
    window w90 as (partition by event_region order by news_date
                   rows between 90 preceding and 1 preceding)
)

select
    concat_ws('|', s.event_region, cast(s.news_date as string)) as row_key,
    s.news_date,
    s.event_region,
    s.stories,
    round(s.severity_total, 1)                                         as severity_total,
    round((s.severity_total - s.severity_90d_avg) / nullif(s.severity_90d_std, 0), 2) as risk_zscore,
    coalesce(v.is_data_gap, false)                                     as is_data_gap,
    v.gdelt_events
from scored s
left join volume_flag v on s.news_date = v.news_date
""",
"models/intermediate/_intermediate.yml": """version: 2

models:
  - name: int_gdelt__events_clean
    description: GDELT events published within 7 days of the event date, with event region.
  - name: int_gdelt__stories
    description: One row per news article (story), grouping all GDELT events from the same URL; rule-based category and severity score. Incremental (last 3 days reprocessed).
    columns:
      - name: story_id
        tests: [unique, not_null]
      - name: category_rule
        tests:
          - accepted_values:
              arguments:
                values: ['vessel_attack_or_seizure', 'infrastructure_strike', 'chokepoint', 'sanctions', 'opec_plus', 'talks_or_ceasefire', 'other']
""",
"models/marts/_marts_events.yml": """version: 2

models:
  - name: fct_risk_index_daily
    description: Daily geopolitical risk index by event region and global (stories, summed severity, 90-day z-score) with a GDELT data-gap flag. Source The GDELT Project.
    columns:
      - name: row_key
        tests: [unique, not_null]
""",
}

import os
for path, content in files.items():
    full = f"{DBT}/{path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "int_gdelt__events_clean int_gdelt__stories fct_risk_index_daily")

display(spark.sql("""
    SELECT news_date, stories, severity_total, risk_zscore, is_data_gap
    FROM oil_shock.gold.fct_risk_index_daily
    WHERE event_region = 'Gulf' AND news_date >= '2026-01-01'
    ORDER BY risk_zscore DESC NULLS LAST LIMIT 10
"""))

display(spark.sql("""
    SELECT category_rule, COUNT(*) AS stories, ROUND(AVG(severity_score), 1) AS avg_severity
    FROM oil_shock.silver.int_gdelt__stories
    GROUP BY category_rule ORDER BY stories DESC
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

stories_sql = open(f"{DBT}/models/intermediate/int_gdelt__stories.sql").read()

old_case = """        when event_root_code in ('03', '04', '05', '06', '08') or url_lc rlike 'ceasefire|truce|talks|negotiat'
            then 'talks_or_ceasefire'
        else 'other'
    end as category_rule,"""
new_case = """        when url_lc rlike 'ceasefire|truce|talks|negotiat|peace-deal|agreement'
            then 'talks_or_ceasefire'
        when event_root_code in ('03', '04', '05', '06', '08')
            then 'diplomacy_general'
        else 'other'
    end as category_rule,"""
assert old_case in stories_sql, "Could not find the category block to replace"
stories_sql = stories_sql.replace(old_case, new_case)

old_tail = """    , 2) as severity_score
from joined
"""
new_tail = """    , 2) as severity_score,
    has_oil_word
      or url_lc rlike 'tanker|vessel|refiner|pipeline|hormuz|red-sea|bab-el|suez|sanction|embargo|opec'
      as is_oil_relevant
from joined
"""
assert old_tail in stories_sql, "Could not find the end of the model"
stories_sql = stories_sql.replace(old_tail, new_tail)

files = {
"models/intermediate/int_gdelt__stories.sql": stories_sql,
"models/marts/fct_risk_index_daily.sql": """with s as (
    select news_date, event_region, severity_score, is_oil_relevant from {{ ref('int_gdelt__stories') }}
    union all
    select news_date, 'Global', severity_score, is_oil_relevant from {{ ref('int_gdelt__stories') }}
),

daily as (
    select
        news_date,
        event_region,
        count(*)                                                    as stories_all,
        sum(severity_score)                                         as severity_all,
        sum(case when is_oil_relevant then 1 else 0 end)            as stories_oil,
        sum(case when is_oil_relevant then severity_score else 0 end) as severity_oil
    from s
    group by news_date, event_region
),

volume as (
    select news_date, count(*) as gdelt_events
    from {{ ref('stg_gdelt__events') }}
    group by news_date
),

volume_flag as (
    select
        news_date,
        gdelt_events,
        gdelt_events < 0.5 * percentile_approx(gdelt_events, 0.5)
            over (order by news_date rows between 30 preceding and 1 preceding) as is_data_gap
    from volume
),

scored as (
    select
        d.*,
        avg(severity_oil)    over w90 as oil_90d_avg,
        stddev(severity_oil) over w90 as oil_90d_std,
        avg(severity_all)    over w90 as all_90d_avg,
        stddev(severity_all) over w90 as all_90d_std
    from daily d
    window w90 as (partition by event_region order by news_date
                   rows between 90 preceding and 1 preceding)
)

select
    concat_ws('|', s.event_region, cast(s.news_date as string)) as row_key,
    s.news_date,
    s.event_region,
    s.stories_oil,
    round(s.severity_oil, 1)                                                  as severity_oil,
    round((s.severity_oil - s.oil_90d_avg) / nullif(s.oil_90d_std, 0), 2)     as oil_risk_zscore,
    s.stories_all,
    round(s.severity_all, 1)                                                  as severity_all,
    round((s.severity_all - s.all_90d_avg) / nullif(s.all_90d_std, 0), 2)     as broad_risk_zscore,
    coalesce(v.is_data_gap, false)                                            as is_data_gap,
    v.gdelt_events
from scored s
left join volume_flag v on s.news_date = v.news_date
""",
"models/intermediate/_intermediate.yml": """version: 2

models:
  - name: int_gdelt__events_clean
    description: GDELT events published within 7 days of the event date, with event region.
  - name: int_gdelt__stories
    description: One row per news article (story), grouping all GDELT events from the same URL; rule-based category, severity score and oil-relevance flag. Incremental (last 3 days reprocessed).
    columns:
      - name: story_id
        tests: [unique, not_null]
      - name: category_rule
        tests:
          - accepted_values:
              arguments:
                values: ['vessel_attack_or_seizure', 'infrastructure_strike', 'chokepoint', 'sanctions', 'opec_plus', 'talks_or_ceasefire', 'diplomacy_general', 'other']
""",
}

for path, content in files.items():
    with open(f"{DBT}/{path}", "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

In [0]:
run_dbt("build", "--select", "int_gdelt__stories fct_risk_index_daily", "--full-refresh")

display(spark.sql("""
    SELECT category_rule, COUNT(*) AS stories,
           SUM(CASE WHEN is_oil_relevant THEN 1 ELSE 0 END) AS oil_relevant,
           ROUND(AVG(severity_score), 1) AS avg_severity
    FROM oil_shock.silver.int_gdelt__stories
    GROUP BY category_rule ORDER BY stories DESC
"""))

display(spark.sql("""
    SELECT news_date, stories_oil, severity_oil, oil_risk_zscore, broad_risk_zscore, is_data_gap
    FROM oil_shock.gold.fct_risk_index_daily
    WHERE event_region = 'Gulf' AND news_date >= '2026-01-01'
    ORDER BY oil_risk_zscore DESC NULLS LAST LIMIT 10
"""))

In [0]:
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"
path = f"{DBT}/models/intermediate/int_gdelt__stories.sql"

sql = open(path).read()
old = """    has_oil_word
      or url_lc rlike 'tanker|vessel|refiner|pipeline|hormuz|red-sea|bab-el|suez|sanction|embargo|opec'
      as is_oil_relevant"""
new = """    has_oil_word
      or category_rule in ('vessel_attack_or_seizure', 'infrastructure_strike', 'chokepoint', 'sanctions', 'opec_plus')
      or url_lc rlike 'tanker|vessel|refiner|pipeline|hormuz|red-sea|bab-el|suez|sanction|embargo|opec'
      as is_oil_relevant"""
assert old in sql, "Could not find the oil-relevance block"
open(path, "w").write(sql.replace(old, new))
print("Updated: dbt/models/intermediate/int_gdelt__stories.sql")

run_dbt("build", "--select", "int_gdelt__stories fct_risk_index_daily", "--full-refresh")

display(spark.sql("""
    SELECT category_rule, COUNT(*) AS stories,
           SUM(CASE WHEN is_oil_relevant THEN 1 ELSE 0 END) AS oil_relevant
    FROM oil_shock.silver.int_gdelt__stories
    GROUP BY category_rule ORDER BY stories DESC
"""))

In [0]:
yml = """version: 2

seeds:
  - name: ref_event_timeline
    description: "Curated, source-linked oil market events (Jan 2025 onward). Hand-picked with hindsight; severity is judgment, display only, never a model input."
    config:
      column_types:
        event_date: date
        severity_1to5: int
    columns:
      - name: event_date
        data_tests:
          - not_null
      - name: event
        data_tests:
          - not_null
      - name: category
        data_tests:
          - not_null
          - accepted_values:
              arguments:
                values: ['sanctions', 'talks_or_ceasefire', 'infrastructure_strike', 'military_escalation', 'chokepoint', 'opec_plus', 'vessel_attack_or_seizure', 'policy', 'stock_release', 'storm', 'other']
      - name: severity_1to5
        data_tests:
          - not_null
          - accepted_values:
              arguments:
                values: [1, 2, 3, 4, 5]
                quote: false
      - name: source_url
        data_tests:
          - not_null
"""

path = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/dbt/seeds/ref_event_timeline.yml"
with open(path, "w") as f:
    f.write(yml)
print("Written:", path)

In [0]:
run_dbt("seed", "--select", "ref_event_timeline")
run_dbt("test", "--select", "ref_event_timeline")

In [0]:
display(spark.sql("""
    SELECT t.region, COUNT(*) AS events,
           MAX(CASE WHEN r.event_region IS NOT NULL THEN 'match' ELSE 'NO MATCH' END) AS status
    FROM oil_shock.silver.ref_event_timeline t
    LEFT JOIN oil_shock.gold.dim_event_region r ON t.region = r.event_region
    GROUP BY t.region
    ORDER BY status, events DESC
"""))

In [0]:
run_dbt("build", "--select", "dim_event_region")

In [0]:
run_dbt("build", "--select", "fct_risk_index_daily")

In [0]:
sql = """with spine as (
    select date_day, day_of_week, is_driving_season, is_heating_season,
           is_hurricane_season, is_war_period
    from {{ ref('dim_date') }}
    where date_day between '2015-01-01' and current_date()
),

px as (
    select price_date, brent, wti, brent_wti_spread, gasoline_crack, diesel_crack, jet_crack
    from {{ ref('fct_spread_daily') }}
),

hh as (
    select price_date, price as henry_hub
    from {{ ref('fct_price_daily') }}
    where series_id = 'RNGWHHD'
),

macro as (
    select date_add(obs_date, 1) as avail_date,
           usd_index, fed_funds_rate, treasury_10y_yield, vix_model_only
    from {{ ref('fct_macro_daily') }}
),

pos_ranked as (
    select date_add(report_date, 3) as avail_date, contract_key,
           managed_money_net_pct_oi, crowding_zscore,
           row_number() over (partition by contract_key, report_date
                              order by case when report_type = 'futures_only' then 1 else 2 end) as rn
    from {{ ref('fct_positioning_weekly') }}
    where contract_key in ('brent_ice', 'wti_nymex')
),

pos as (
    select avail_date,
           max(case when contract_key = 'brent_ice' then managed_money_net_pct_oi end) as brent_mm_net_pct_oi,
           max(case when contract_key = 'brent_ice' then crowding_zscore end)          as brent_crowding_z,
           max(case when contract_key = 'wti_nymex' then managed_money_net_pct_oi end) as wti_mm_net_pct_oi,
           max(case when contract_key = 'wti_nymex' then crowding_zscore end)          as wti_crowding_z
    from pos_ranked
    where rn = 1
    group by avail_date
),

choke as (
    select date_add(transit_date, 7) as avail_date,
           max(case when lower(chokepoint_name) like '%hormuz%'    then ships_tanker_pct_of_2025 end) as hormuz_tanker_pct,
           max(case when lower(chokepoint_name) like '%hormuz%'    then ships_total_pct_of_2025 end)  as hormuz_total_pct,
           max(case when lower(chokepoint_name) like '%mandeb%'    then ships_tanker_pct_of_2025 end) as mandeb_tanker_pct,
           max(case when lower(chokepoint_name) like '%suez%'      then ships_tanker_pct_of_2025 end) as suez_tanker_pct,
           max(case when lower(chokepoint_name) like '%good hope%' then ships_tanker_pct_of_2025 end) as cape_tanker_pct
    from {{ ref('fct_chokepoint_daily') }}
    group by transit_date
),

risk as (
    select news_date as avail_date,
           max(case when event_region = 'Gulf' then oil_risk_zscore end)             as gulf_risk_surprise_z,
           max(case when event_region = 'Gulf' then oil_risk_level_index end)        as gulf_risk_level_index,
           max(case when event_region = 'Gulf' then talks_zscore end)                as gulf_talks_z,
           max(case when event_region = 'Global' then oil_risk_zscore end)           as global_risk_surprise_z,
           max(case when event_region = 'Red Sea' then oil_risk_zscore end)          as redsea_risk_surprise_z,
           max(case when event_region = 'Russia/Black Sea' then oil_risk_zscore end) as russia_risk_surprise_z,
           max(cast(is_data_gap as int)) = 1                                         as risk_data_gap
    from {{ ref('fct_risk_index_daily') }}
    group by news_date
),

us as (
    select date_add(week_ending, 5) as avail_date,
           crude_stocks_commercial_kbbl, crude_stocks_change_kbbl,
           refinery_utilization_pct, crude_production_kbd
    from {{ ref('fct_us_weekly_balance') }}
),

storm as (
    select storm_date, storms_in_gulf, hurricane_in_gulf
    from {{ ref('fct_gulf_storm_daily') }}
),

mm as (
    select date_add(add_months(month_start, 1), 14) as avail_date,
           us_inflation_yoy_pct, us_industrial_production_yoy_pct
    from {{ ref('fct_macro_monthly') }}
),

cli as (
    select date_add(add_months(period_date, 1), 12) as avail_date,
           max(case when area_code = 'CHN' then cli_value end)     as china_cli,
           max(case when area_code = 'CHN' then cli_change_3m end) as china_cli_change_3m,
           max(case when area_code = 'USA' then cli_value end)     as us_cli,
           max(case when area_code = 'USA' then cli_change_3m end) as us_cli_change_3m
    from {{ ref('fct_leading_indicator_monthly') }}
    group by period_date
),

joined as (
    select sp.*,
           px.brent, px.wti, px.brent_wti_spread, px.gasoline_crack, px.diesel_crack, px.jet_crack,
           hh.henry_hub,
           macro.usd_index, macro.fed_funds_rate, macro.treasury_10y_yield, macro.vix_model_only,
           pos.brent_mm_net_pct_oi, pos.brent_crowding_z, pos.wti_mm_net_pct_oi, pos.wti_crowding_z,
           choke.hormuz_tanker_pct, choke.hormuz_total_pct, choke.mandeb_tanker_pct,
           choke.suez_tanker_pct, choke.cape_tanker_pct,
           risk.gulf_risk_surprise_z, risk.gulf_risk_level_index, risk.gulf_talks_z,
           risk.global_risk_surprise_z, risk.redsea_risk_surprise_z, risk.russia_risk_surprise_z,
           risk.risk_data_gap,
           us.crude_stocks_commercial_kbbl, us.crude_stocks_change_kbbl,
           us.refinery_utilization_pct, us.crude_production_kbd,
           coalesce(storm.storms_in_gulf, 0)        as storms_in_gulf,
           coalesce(storm.hurricane_in_gulf, false) as hurricane_in_gulf,
           mm.us_inflation_yoy_pct, mm.us_industrial_production_yoy_pct,
           cli.china_cli, cli.china_cli_change_3m, cli.us_cli, cli.us_cli_change_3m
    from spine sp
    left join px    on px.price_date = sp.date_day
    left join hh    on hh.price_date = sp.date_day
    left join macro on macro.avail_date = sp.date_day
    left join pos   on pos.avail_date = sp.date_day
    left join choke on choke.avail_date = sp.date_day
    left join risk  on risk.avail_date = sp.date_day
    left join us    on us.avail_date = sp.date_day
    left join storm on storm.storm_date = sp.date_day
    left join mm    on mm.avail_date = sp.date_day
    left join cli   on cli.avail_date = sp.date_day
),

filled as (
    select
        date_day, day_of_week, is_driving_season, is_heating_season, is_hurricane_season, is_war_period,
        brent, wti, brent_wti_spread, gasoline_crack, diesel_crack, jet_crack, henry_hub,
        last_value(usd_index, true)                         over w as usd_index,
        last_value(fed_funds_rate, true)                    over w as fed_funds_rate,
        last_value(treasury_10y_yield, true)                over w as treasury_10y_yield,
        last_value(vix_model_only, true)                    over w as vix_model_only,
        last_value(brent_mm_net_pct_oi, true)               over w as brent_mm_net_pct_oi,
        last_value(brent_crowding_z, true)                  over w as brent_crowding_z,
        last_value(wti_mm_net_pct_oi, true)                 over w as wti_mm_net_pct_oi,
        last_value(wti_crowding_z, true)                    over w as wti_crowding_z,
        last_value(hormuz_tanker_pct, true)                 over w as hormuz_tanker_pct,
        last_value(hormuz_total_pct, true)                  over w as hormuz_total_pct,
        last_value(mandeb_tanker_pct, true)                 over w as mandeb_tanker_pct,
        last_value(suez_tanker_pct, true)                   over w as suez_tanker_pct,
        last_value(cape_tanker_pct, true)                   over w as cape_tanker_pct,
        gulf_risk_surprise_z,
        max(gulf_risk_surprise_z)   over w3 as gulf_risk_surprise_z_3d_max,
        gulf_risk_level_index,
        gulf_talks_z,
        max(gulf_talks_z)           over w3 as gulf_talks_z_3d_max,
        global_risk_surprise_z,
        max(global_risk_surprise_z) over w3 as global_risk_surprise_z_3d_max,
        max(redsea_risk_surprise_z) over w3 as redsea_risk_surprise_z_3d_max,
        max(russia_risk_surprise_z) over w3 as russia_risk_surprise_z_3d_max,
        coalesce(risk_data_gap, false) as risk_data_gap,
        last_value(crude_stocks_commercial_kbbl, true)      over w as crude_stocks_commercial_kbbl,
        last_value(crude_stocks_change_kbbl, true)          over w as crude_stocks_change_kbbl,
        last_value(refinery_utilization_pct, true)          over w as refinery_utilization_pct,
        last_value(crude_production_kbd, true)              over w as us_crude_production_kbd,
        storms_in_gulf,
        hurricane_in_gulf,
        last_value(us_inflation_yoy_pct, true)              over w as us_inflation_yoy_pct,
        last_value(us_industrial_production_yoy_pct, true)  over w as us_industrial_production_yoy_pct,
        last_value(china_cli, true)                         over w as china_cli,
        last_value(china_cli_change_3m, true)               over w as china_cli_change_3m,
        last_value(us_cli, true)                            over w as us_cli,
        last_value(us_cli_change_3m, true)                  over w as us_cli_change_3m
    from joined
    window w  as (order by date_day rows between unbounded preceding and current row),
           w3 as (order by date_day rows between 2 preceding and current row)
),

trading as (
    select *,
           ln(brent / lag(brent) over (order by date_day))    as brent_ret_1d,
           ln(brent / lag(brent, 5) over (order by date_day)) as brent_ret_5d_past
    from filled
    where brent is not null
)

select
    date_day as price_date,
    *except (date_day),
    stddev(brent_ret_1d) over (order by date_day rows between 20 preceding and 1 preceding) as brent_vol_20d
from trading
"""

yml = """version: 2

models:
  - name: fct_model_features_daily
    description: "One row per Brent trading day (2015+). Point-in-time features: each value appears only from the day it was published (macro +1d, COT +3d, EIA weekly +5d, PortWatch +7d, monthly macro and OECD CLI ~+6 weeks), then carried forward. Risk values also as 3-day max so weekend news reaches Monday."
    columns:
      - name: price_date
        data_tests:
          - unique
          - not_null
      - name: brent
        data_tests:
          - not_null
"""

base = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/dbt/models/marts/"
with open(base + "fct_model_features_daily.sql", "w") as f:
    f.write(sql)
with open(base + "fct_model_features_daily.yml", "w") as f:
    f.write(yml)
print("Written both files.")

In [0]:
run_dbt("build", "--select", "fct_model_features_daily")

In [0]:
from pyspark.sql import functions as F

df = spark.table("oil_shock.gold.fct_model_features_daily")
cols = [c for c in df.columns if c != "price_date"]
recent = F.col("price_date") >= F.lit("2025-01-01")

aggs = [F.count("*").alias("total_rows"),
        F.sum(F.when(recent, 1).otherwise(0)).alias("recent_rows")]
for c in cols:
    aggs += [F.min(F.when(F.col(c).isNotNull(), F.col("price_date"))).alias(f"{c}__first"),
             F.max(F.when(F.col(c).isNotNull(), F.col("price_date"))).alias(f"{c}__last"),
             F.sum(F.when(recent & F.col(c).isNotNull(), 1).otherwise(0)).alias(f"{c}__filled")]

r = df.agg(*aggs).first()
n_recent = r["recent_rows"]
rows = [(c, str(r[f"{c}__first"]), str(r[f"{c}__last"]),
         round(100 * r[f"{c}__filled"] / n_recent, 1)) for c in cols]

print("Total rows:", r["total_rows"], "| rows since 2025:", n_recent)
display(spark.createDataFrame(rows, ["column", "first_date", "last_date", "pct_filled_since_2025"])
        .orderBy("pct_filled_since_2025"))

In [0]:
sql = """with cats as (
    select explode(array('vessel_attack_or_seizure', 'infrastructure_strike', 'chokepoint',
                         'sanctions', 'opec_plus', 'talks_or_ceasefire')) as category
),

stories as (
    select news_date, event_region, category_rule as category, severity_score
    from {{ ref('int_gdelt__stories') }}
    union all
    select news_date, 'Global' as event_region, category_rule as category, severity_score
    from {{ ref('int_gdelt__stories') }}
),

daily as (
    select news_date, event_region, category,
           count(*)            as stories,
           sum(severity_score) as severity
    from stories
    where category in (select category from cats)
    group by news_date, event_region, category
),

days as (
    select news_date, is_data_gap
    from {{ ref('fct_risk_index_daily') }}
    where event_region = 'Global'
),

spine as (
    select d.news_date, d.is_data_gap, r.event_region, c.category
    from days d
    cross join {{ ref('dim_event_region') }} r
    cross join cats c
),

filled as (
    select sp.news_date, sp.event_region, sp.category, sp.is_data_gap,
           case when sp.is_data_gap then null else coalesce(dl.stories, 0) end  as stories,
           case when sp.is_data_gap then null else coalesce(dl.severity, 0) end as severity
    from spine sp
    left join daily dl
      on dl.news_date = sp.news_date
     and dl.event_region = sp.event_region
     and dl.category = sp.category
),

scored as (
    select f.*,
           avg(ln(1 + stories))    over w as log_avg_90d,
           stddev(ln(1 + stories)) over w as log_std_90d
    from filled f
    window w as (partition by event_region, category order by news_date
                 rows between 90 preceding and 1 preceding)
)

select
    concat_ws('|', event_region, category, cast(news_date as string))         as row_key,
    news_date,
    event_region,
    category,
    stories,
    round(severity, 1)                                                         as severity,
    round((ln(1 + stories) - log_avg_90d) / nullif(log_std_90d, 0), 2)         as stories_zscore,
    is_data_gap
from scored
"""

yml = """version: 2

models:
  - name: fct_news_category_daily
    description: "Daily GDELT story counts by region and story type, with a surprise score (z of log(1+count) vs prior 90 days). NULL on data-gap days. Used by driver attribution v2."
    columns:
      - name: row_key
        data_tests:
          - unique
          - not_null
      - name: category
        data_tests:
          - accepted_values:
              arguments:
                values: ['vessel_attack_or_seizure', 'infrastructure_strike', 'chokepoint', 'sanctions', 'opec_plus', 'talks_or_ceasefire']
"""

base = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/dbt/models/marts/"
with open(base + "fct_news_category_daily.sql", "w") as f:
    f.write(sql)
with open(base + "fct_news_category_daily.yml", "w") as f:
    f.write(yml)
print("Written both files.")

In [0]:
run_dbt("build", "--select", "fct_news_category_daily")

In [0]:
path = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/dbt/seeds/ref_event_timeline.csv"
line = ('2026-02-17,"Iran temporarily closes parts of the Strait of Hormuz for live-fire drills as nuclear talks '
        'with the US begin",chokepoint,Gulf,3,verified,'
        'https://www.pbs.org/newshour/world/iran-temporarily-closing-strait-of-hormuz-for-live-fire-drills-as-nuclear-talks-with-u-s-begin\n')

with open(path) as f:
    content = f.read()
if "2026-02-17," in content:
    print("Already there, nothing added.")
else:
    with open(path, "a") as f:
        if not content.endswith("\n"):
            f.write("\n")
        f.write(line)
    print("Added. Rows now:", sum(1 for _ in open(path)) - 1)

run_dbt("seed", "--select", "ref_event_timeline")

In [0]:
import os

root = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/dbt/models/"
staging = root + "staging/"
marts = root + "marts/"
print("staging folder exists:", os.path.isdir(staging), "| marts folder exists:", os.path.isdir(marts))

sources_yml = """version: 2

sources:
  - name: bronze_archive
    database: oil_shock
    schema: bronze
    tables:
      - name: steo_archive
        description: "EIA STEO monthly releases 2015-01 to 2026-09 (Brent BREPUUS, WTI WTIPUUS), parsed from the archived Excel files. One-time backfill by notebooks/01_ingestion/19_steo_archive."
"""

stg_sql = """select
    release_month,
    series_id,
    to_date(concat(target_month, '-01')) as target_month,
    value,
    source_file
from {{ source('bronze_archive', 'steo_archive') }}
where value is not null
"""

stg_yml = """version: 2

models:
  - name: stg_eia__steo_archive
    description: "Archived EIA STEO releases (Brent, WTI), target month as date."
    columns:
      - name: release_month
        data_tests:
          - not_null
      - name: target_month
        data_tests:
          - not_null
"""

fct_sql = """with api as (
    select
        row_key,
        steo_vintage    as release_month,
        series_id,
        series_label,
        units,
        period_date     as target_month,
        value,
        'api'           as source
    from {{ ref('stg_eia__series') }}
    where data_kind = 'forecast'
),

labels as (
    select series_id, max(series_label) as series_label, max(units) as units
    from api
    group by series_id
),

archive as (
    select
        concat_ws('|', 'steo_archive', a.release_month, a.series_id, cast(a.target_month as string)) as row_key,
        a.release_month,
        a.series_id,
        l.series_label,
        l.units,
        a.target_month,
        a.value,
        'steo_archive'  as source
    from {{ ref('stg_eia__steo_archive') }} a
    left join labels l on l.series_id = a.series_id
    where a.release_month not in (select distinct release_month from api)
),

unioned as (
    select * from api
    union all
    select * from archive
)

select
    row_key,
    release_month,
    series_id,
    series_label,
    units,
    target_month,
    value,
    target_month >= to_date(concat(release_month, '-01'))                    as is_forecast,
    months_between(target_month, to_date(concat(release_month, '-01')))     as months_ahead,
    source
from unioned
"""

files = {staging + "_steo_archive_sources.yml": sources_yml,
         staging + "stg_eia__steo_archive.sql": stg_sql,
         staging + "stg_eia__steo_archive.yml": stg_yml,
         marts + "fct_steo_forecast.sql": fct_sql}
for path, text in files.items():
    with open(path, "w") as f:
        f.write(text)
    print("Written:", path.replace(root, "models/"))

In [0]:
run_dbt("build", "--select", "stg_eia__steo_archive+")

In [0]:
# Rebuild the STEO forecast mart so the new EIA series reach gold, then check them
run_dbt("build", "--select", "stg_eia__series fct_steo_forecast")

chk = spark.sql("""
    SELECT series_id, release_month, COUNT(*) AS rows,
           MIN(target_month) AS first_month, MAX(target_month) AS last_month,
           ROUND(MAX(CASE WHEN target_month = '2026-09-01' THEN value END), 2) AS sep_2026
    FROM oil_shock.gold.fct_steo_forecast
    WHERE release_month = (SELECT MAX(release_month) FROM oil_shock.gold.fct_steo_forecast)
      AND (series_id LIKE 'COPR_%' OR series_id LIKE 'COPS_%' OR series_id IN ('PAPR_TC', 'PAPR_QA'))
    GROUP BY series_id, release_month
    ORDER BY series_id
""").toPandas()
print(chk.to_string(index=False))
print(f"\nSeries found: {len(chk)} (expected 12)")

In [0]:
# Round 2: PortWatch ports -> dbt (seed + staging + mart), then build only these models
import os
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

files = {
"seeds/ref_gulf_ports.csv": """port_name,iso3,role,chart_label
Juaymah,SAU,inside_hormuz,Juaymah (Saudi Arabia)
Ras Tanura,SAU,inside_hormuz,Ras Tanura (Saudi Arabia)
Basrah Oil Terminal,IRQ,inside_hormuz,Basrah (Iraq)
Mina Al Ahmadi,KWT,inside_hormuz,Mina Al Ahmadi (Kuwait)
Jabal Az Zannah-Ruways,ARE,inside_hormuz,Ruwais (UAE)
Das Island,ARE,inside_hormuz,Das Island (UAE)
Yanbu (King Fahd Port),SAU,bypass,Yanbu (Saudi Red Sea)
Rabigh,SAU,bypass,Rabigh (Saudi Red Sea)
Fujairah,ARE,bypass,Fujairah (UAE)
Port of Sohar,OMN,outside_hormuz,Sohar (Oman)
Ras Laffan,QAT,gas,Ras Laffan (Qatar)
Bandar-E Pars Terminal,IRN,gas,Bandar-e Pars (Iran)
Kharg Island,IRN,tracking_gap,Kharg Island (Iran)
""",
"models/staging/stg_portwatch__ports.sql": """select
    concat_ws('|', portid, date)                                               as row_key,
    to_date(date)                                                              as port_date,
    portid                                                                     as port_id,
    portname                                                                   as port_name,
    iso3,
    cast(get_json_object(attributes_json, '$.portcalls_tanker') as double)     as tanker_calls,
    cast(get_json_object(attributes_json, '$.export_tanker') as double)        as export_tanker_t,
    cast(get_json_object(attributes_json, '$.import_tanker') as double)        as import_tanker_t,
    cast(get_json_object(attributes_json, '$.portcalls') as double)            as port_calls_all,
    loaded_at
from {{ source('bronze', 'portwatch_ports') }}
""",
"models/staging/stg_portwatch__ports.yml": """version: 2

models:
  - name: stg_portwatch__ports
    description: Daily port calls and trade estimates (metric tons) for Gulf ports, from ship tracking. Sources UN Global Platform; IMF PortWatch.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: port_date
        tests: [not_null]
""",
"models/marts/fct_port_daily.sql": """with base as (
    select
        s.port_id, s.port_name, s.iso3, s.port_date,
        s.tanker_calls, s.export_tanker_t, s.import_tanker_t,
        avg(s.export_tanker_t) over w7 as export_tanker_t_7d_avg,
        avg(s.tanker_calls)    over w7 as tanker_calls_7d_avg
    from {{ ref('stg_portwatch__ports') }} s
    window w7 as (partition by s.port_id order by s.port_date rows between 6 preceding and current row)
),

normal_2025 as (
    select port_id, avg(export_tanker_t) as export_tanker_t_2025_avg
    from {{ ref('stg_portwatch__ports') }}
    where year(port_date) = 2025
    group by port_id
)

select
    concat_ws('|', b.port_id, cast(b.port_date as string))                          as row_key,
    b.*,
    coalesce(r.role, 'other')                                                        as port_role,
    coalesce(r.chart_label, b.port_name)                                             as chart_label,
    n.export_tanker_t_2025_avg,
    round(100 * b.export_tanker_t_7d_avg / nullif(n.export_tanker_t_2025_avg, 0), 1) as export_pct_of_2025,
    -- rough barrels: ~7.33 barrels per metric ton of crude; only meaningful for crude ports, not gas terminals
    round(b.export_tanker_t_7d_avg * 7.33 / 1000, 1)                                 as export_kbd_est
from base b
left join normal_2025 n using (port_id)
left join {{ ref('ref_gulf_ports') }} r
  on r.port_name = b.port_name and r.iso3 = b.iso3
""",
"models/marts/_marts_ports.yml": """version: 2

models:
  - name: fct_port_daily
    description: Daily tanker exports per Gulf port (metric tons, 7-day average, percent of the 2025 average, rough barrels per day) with a role flag (inside Hormuz, bypass, gas, tracking gap). Ship-tracking estimates, not customs data. Sources UN Global Platform; IMF PortWatch.
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: port_role
        tests:
          - accepted_values:
              arguments:
                values: ['inside_hormuz', 'bypass', 'outside_hormuz', 'gas', 'tracking_gap', 'other']
""",
}
for path, content in files.items():
    with open(f"{DBT}/{path}", "w") as f:
        f.write(content)
    print("Written:", f"dbt/{path}")

# Register the new bronze table as a dbt source (adds one entry; leaves the rest of the file unchanged)
src_path = f"{DBT}/models/staging/_sources.yml"
src = open(src_path).read()
if "name: portwatch_ports\n" not in src:
    anchor = "      - name: portwatch_chokepoints\n        description: IMF PortWatch daily chokepoint transits. Sources UN Global Platform; IMF PortWatch.\n"
    assert anchor in src, "Could not find the portwatch_chokepoints entry in _sources.yml"
    src = src.replace(anchor, anchor + "      - name: portwatch_ports\n"
                      "        description: IMF PortWatch daily port calls and trade estimates for Gulf ports. Sources UN Global Platform; IMF PortWatch.\n")
    open(src_path, "w").write(src)
    print("Updated: dbt/models/staging/_sources.yml (added portwatch_ports)")
else:
    print("Already registered: portwatch_ports")

# Seed project config: keep the seed's text columns as text
proj_path = f"{DBT}/dbt_project.yml"
proj = open(proj_path).read()
if "ref_gulf_ports:" not in proj:
    proj = proj.rstrip("\n") + "\n    ref_gulf_ports:\n      +column_types:\n        port_name: string\n        iso3: string\n        role: string\n        chart_label: string\n"
    open(proj_path, "w").write(proj)
    print("Updated: dbt/dbt_project.yml (ref_gulf_ports column types)")

ok = run_dbt("build", "--select", "ref_gulf_ports stg_portwatch__ports+")

if ok:
    display(spark.sql("""
        SELECT chart_label, port_role, MAX(port_date) AS last_day,
               ROUND(MAX(export_tanker_t_2025_avg))                                   AS t_per_day_2025,
               MAX_BY(export_pct_of_2025, port_date)                                  AS latest_pct_of_2025,
               MAX_BY(export_kbd_est, port_date)                                      AS latest_kbd_est
        FROM oil_shock.gold.fct_port_daily
        WHERE port_role <> 'other'
        GROUP BY chart_label, port_role
        ORDER BY t_per_day_2025 DESC"""))

In [0]:
# Where do the port export estimates stop being complete? Total Gulf tanker exports per day, last 30 days
display(spark.sql("""
    SELECT port_date,
           ROUND(SUM(export_tanker_t))                                   AS total_export_t,
           SUM(CASE WHEN export_tanker_t > 0 THEN 1 ELSE 0 END)          AS ports_with_exports,
           ROUND(SUM(tanker_calls), 1)                                   AS total_tanker_calls,
           ROUND(MAX(CASE WHEN port_name = 'Yanbu (King Fahd Port)' THEN export_tanker_t END)) AS yanbu_t
    FROM oil_shock.gold.fct_port_daily
    WHERE port_date >= date_sub(DATE '2026-10-02', 30)
    GROUP BY port_date
    ORDER BY port_date DESC"""))

In [0]:
# Monthly tanker exports by port (thousand metric tons per day, average over the month), 2025 vs 2026
display(spark.sql("""
    SELECT chart_label, port_role,
           ROUND(AVG(CASE WHEN year(port_date) = 2025 THEN export_tanker_t END) / 1000)               AS avg_2025,
           ROUND(AVG(CASE WHEN date_format(port_date, 'yyyy-MM') = '2026-03' THEN export_tanker_t END) / 1000) AS mar,
           ROUND(AVG(CASE WHEN date_format(port_date, 'yyyy-MM') = '2026-04' THEN export_tanker_t END) / 1000) AS apr,
           ROUND(AVG(CASE WHEN date_format(port_date, 'yyyy-MM') = '2026-05' THEN export_tanker_t END) / 1000) AS may,
           ROUND(AVG(CASE WHEN date_format(port_date, 'yyyy-MM') = '2026-06' THEN export_tanker_t END) / 1000) AS jun,
           ROUND(AVG(CASE WHEN date_format(port_date, 'yyyy-MM') = '2026-07' THEN export_tanker_t END) / 1000) AS jul,
           ROUND(AVG(CASE WHEN date_format(port_date, 'yyyy-MM') = '2026-08' THEN export_tanker_t END) / 1000) AS aug,
           ROUND(AVG(CASE WHEN date_format(port_date, 'yyyy-MM') = '2026-09' THEN export_tanker_t END) / 1000) AS sep
    FROM oil_shock.gold.fct_port_daily
    WHERE port_role <> 'other'
    GROUP BY chart_label, port_role
    ORDER BY avg_2025 DESC"""))

In [0]:
# Paste-and-run: add the Houthi blockade (20 Jul) and the Red Sea tanker attack (24 Aug) to the event seed
path = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/dbt/seeds/ref_event_timeline.csv"
new_lines = [
    ('2026-07-20,"Houthis declare a maritime blockade of Saudi Arabia; ships reroute away from Bab el-Mandeb",'
     'chokepoint,Red Sea,4,verified,'
     'https://www.fdd.org/analysis/2026/07/22/houthis-announce-blockade-of-saudi-arabia-ships-reroute-to-avoid-bab-al-mandeb-strait/\n'),
    ('2026-08-24,"Houthis attack a Saudi oil tanker in the Red Sea; Saudi exports from Yanbu collapse in August",'
     'vessel_attack_or_seizure,Red Sea,4,verified,'
     'https://aljazeera.com/news/2026/8/24/yemens-houthis-report-attack-on-saudi-ship\n'),
]

with open(path) as f:
    content = f.read()
added = 0
with open(path, "a") as f:
    if not content.endswith("\n"):
        f.write("\n")
    for line in new_lines:
        key = line.split('"')[0] + '"' + line.split('"')[1][:20]
        if key in content:
            print("Already there:", line[:40])
        else:
            f.write(line)
            added += 1
print("Added", added, "rows. Rows now:", sum(1 for _ in open(path)) - 1)

run_dbt("seed", "--select", "ref_event_timeline")
run_dbt("test", "--select", "ref_event_timeline")

In [0]:
# New cell in 30_dbt_run - Singapore MPA without the SQL warehouse.
# Writes the dbt files, then builds the same view and table on this notebook's compute.
# Tonight's 21:00 job (dbt_build task) rebuilds both through dbt and runs the formal tests.
# No need to run cells 1 and 3 for this cell.
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"

# 1. Register the new bronze table in _sources.yml (added only once)
p = f"{DBT}/models/staging/_sources.yml"
txt = open(p).read()
if "name: sg_mpa_monthly" in txt:
    print("_sources.yml: sg_mpa_monthly already there")
else:
    anchor = "    tables:\n"
    i = txt.index(anchor) + len(anchor)
    entry = ("      - name: sg_mpa_monthly\n"
             "        description: Singapore MPA monthly bunker sales (thousand tonnes) and vessel arrivals over 75 GT, from data.gov.sg.\n")
    open(p, "w").write(txt[:i] + entry + txt[i:])
    print("_sources.yml: sg_mpa_monthly added")

files = {}

files["models/staging/stg_sg_mpa__monthly.sql"] = """\
-- Singapore MPA monthly data (data.gov.sg), one row per dataset, month, category and measure
select
    concat_ws('|', dataset, month, category, measure)  as row_key,
    dataset,
    to_date(concat(month, '-01'))                       as month_start,
    category,
    measure,
    value,
    loaded_at
from {{ source('bronze', 'sg_mpa_monthly') }}
where value is not null
  and month rlike '^[0-9]{4}-[0-9]{2}$'
"""

files["models/staging/stg_sg_mpa__monthly.yml"] = """\
version: 2

models:
  - name: stg_sg_mpa__monthly
    description: Singapore MPA monthly bunker sales (thousand tonnes, total and by fuel type) and vessel arrivals over 75 GT (count and thousand GT, total and by vessel type).
    columns:
      - name: row_key
        tests: [unique, not_null]
      - name: dataset
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ['bunker_total', 'bunker_by_type', 'arrivals_total', 'arrivals_by_type']
      - name: month_start
        tests: [not_null]
      - name: value
        tests: [not_null]
"""

files["models/marts/fct_sg_bunker_monthly.sql"] = """\
-- Singapore bunker hub, one row per month from 2019: fuel sold, fuel mix, ship arrivals, and both vs the 2025 average
with s as (
    select * from {{ ref('stg_sg_mpa__monthly') }}
    where month_start >= '2019-01-01'
),

m as (
    select
        month_start,
        max(case when dataset = 'bunker_total' then value end)                                   as bunker_sales_kt,
        sum(case when dataset = 'bunker_by_type' and category like '%Low Sulphur Fuel Oil%'
                 then value end)                                                                  as vlsfo_kt,
        sum(case when dataset = 'bunker_by_type' and category like '%Marine Fuel Oil%'
                 then value end)                                                                  as hsfo_kt,
        sum(case when dataset = 'bunker_by_type'
                  and (category like '%Gas Oil%' or category like '%Diesel%') then value end)     as distillates_kt,
        max(case when dataset = 'arrivals_total' and measure = 'vessels' then value end)          as vessel_arrivals,
        max(case when dataset = 'arrivals_total' and measure = 'gross_tonnage_kgt' then value end) as arrivals_gt_kgt,
        max(case when dataset = 'arrivals_by_type' and measure = 'vessels'
                  and category = 'Container' then value end)                                      as container_arrivals,
        max(case when dataset = 'arrivals_by_type' and measure = 'vessels'
                  and category = 'Bulk Carrier' then value end)                                   as bulk_carrier_arrivals,
        max(case when dataset = 'arrivals_by_type' and measure = 'vessels'
                  and category = 'Tanker' then value end)                                         as tanker_arrivals
    from s
    group by month_start
),

base as (
    select avg(bunker_sales_kt) as bunker_2025_avg_kt,
           avg(vessel_arrivals)  as arrivals_2025_avg
    from m
    where year(month_start) = 2025
)

select
    m.month_start,
    m.bunker_sales_kt,
    m.vlsfo_kt,
    m.hsfo_kt,
    m.distillates_kt,
    m.bunker_sales_kt - coalesce(m.vlsfo_kt, 0) - coalesce(m.hsfo_kt, 0)
                      - coalesce(m.distillates_kt, 0)                    as other_fuels_kt,
    m.vessel_arrivals,
    m.arrivals_gt_kgt,
    m.container_arrivals,
    m.bulk_carrier_arrivals,
    m.tanker_arrivals,
    m.bunker_sales_kt * 1000 / nullif(m.vessel_arrivals, 0)              as bunker_t_per_arrival,
    b.bunker_2025_avg_kt,
    b.arrivals_2025_avg,
    m.bunker_sales_kt / nullif(b.bunker_2025_avg_kt, 0)                  as bunker_pct_of_2025,
    m.vessel_arrivals / nullif(b.arrivals_2025_avg, 0)                   as arrivals_pct_of_2025
from m
cross join base b
"""

files["models/marts/_marts_bunkers.yml"] = """\
version: 2

models:
  - name: fct_sg_bunker_monthly
    description: Singapore bunker hub by month from 2019. Bunker sales in thousand tonnes (total and by fuel group), vessel arrivals over 75 GT (total, container, bulk carrier, tanker), fuel sold per arriving ship, and each compared with the 2025 monthly average. Source MPA via data.gov.sg.
    columns:
      - name: month_start
        tests: [unique, not_null]
      - name: bunker_sales_kt
        tests: [not_null]
      - name: vessel_arrivals
        tests: [not_null]
"""

for rel, body in files.items():
    with open(f"{DBT}/{rel}", "w") as f:
        f.write(body)
    print("Written:", rel, f"({len(body):,} characters)")

# 2. Build now on notebook compute, using the exact SQL from the dbt files
def compiled(rel):
    sql = open(f"{DBT}/{rel}").read()
    sql = sql.replace("{{ source('bronze', 'sg_mpa_monthly') }}", "oil_shock.bronze.sg_mpa_monthly")
    sql = sql.replace("{{ ref('stg_sg_mpa__monthly') }}", "oil_shock.silver.stg_sg_mpa__monthly")
    return "\n".join(line for line in sql.splitlines() if not line.strip().startswith("--"))

spark.sql("CREATE OR REPLACE VIEW oil_shock.silver.stg_sg_mpa__monthly AS\n" + compiled("models/staging/stg_sg_mpa__monthly.sql"))
print("Built view:  oil_shock.silver.stg_sg_mpa__monthly")
spark.sql("CREATE OR REPLACE TABLE oil_shock.gold.fct_sg_bunker_monthly AS\n" + compiled("models/marts/fct_sg_bunker_monthly.sql"))
print("Built table: oil_shock.gold.fct_sg_bunker_monthly")

# 3. Same checks as the dbt tests (each count should be 0)
checks = {
    "stg row_key duplicates": "SELECT COUNT(*) - COUNT(DISTINCT row_key) FROM oil_shock.silver.stg_sg_mpa__monthly",
    "stg nulls (row_key, dataset, month, value)": """SELECT COUNT(*) FROM oil_shock.silver.stg_sg_mpa__monthly
        WHERE row_key IS NULL OR dataset IS NULL OR month_start IS NULL OR value IS NULL""",
    "stg unexpected dataset names": """SELECT COUNT(*) FROM oil_shock.silver.stg_sg_mpa__monthly
        WHERE dataset NOT IN ('bunker_total', 'bunker_by_type', 'arrivals_total', 'arrivals_by_type')""",
    "fct month duplicates": "SELECT COUNT(*) - COUNT(DISTINCT month_start) FROM oil_shock.gold.fct_sg_bunker_monthly",
    "fct nulls (month, bunker, arrivals)": """SELECT COUNT(*) FROM oil_shock.gold.fct_sg_bunker_monthly
        WHERE month_start IS NULL OR bunker_sales_kt IS NULL OR vessel_arrivals IS NULL""",
}
failed = 0
for name, q in checks.items():
    n = spark.sql(q).collect()[0][0]
    failed += n != 0
    print(f"{'PASS' if n == 0 else 'FAIL'}  {name}: {n}")
print("All checks passed" if not failed else f"{failed} check(s) failed")


display(spark.sql("""
    SELECT month_start, ROUND(bunker_sales_kt) AS bunker_kt, ROUND(vlsfo_kt) AS vlsfo_kt,
           ROUND(hsfo_kt) AS hsfo_kt, ROUND(distillates_kt) AS distillates_kt, ROUND(other_fuels_kt) AS other_kt,
           vessel_arrivals, container_arrivals, bulk_carrier_arrivals,
           ROUND(bunker_t_per_arrival) AS t_per_arrival,
           ROUND(bunker_pct_of_2025 * 100) AS bunker_pct_2025, ROUND(arrivals_pct_of_2025 * 100) AS arrivals_pct_2025
    FROM oil_shock.gold.fct_sg_bunker_monthly
    WHERE month_start >= '2025-01-01'
    ORDER BY month_start DESC
"""))

In [0]:
# Fuel oil demand on the Demand page: dbt file + gold table + check  (one cell, ~1 min)
import time
t0 = time.time()
def log(m): print(f"[{time.time()-t0:5.0f}s] {m}", flush=True)

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT_FILE = f"{REPO}/dbt/models/marts/fct_country_balance_monthly.sql"

DBT_SQL = """with b as (
    select * from {{ ref('fct_oil_balance_monthly') }}
)

select
    concat_ws('|', country_code, cast(period_date as string)) as row_key,
    country_code,
    period_date,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'INDPROD'  then value end) as crude_production_kbd,
    max(case when product_code = 'NGL'      and flow_code = 'INDPROD'  then value end) as ngl_production_kbd,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'TOTIMPSB' then value end) as crude_imports_kbd,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'TOTEXPSB' then value end) as crude_exports_kbd,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'REFINOBS' then value end) as crude_refinery_intake_kbd,
    max(case when product_code = 'TOTPRODS' and flow_code = 'TOTDEMO'  then value end) as product_demand_kbd,
    max(case when product_code = 'TOTPRODS' and flow_code = 'REFGROUT' then value end) as product_output_kbd,
    max(case when product_code = 'TOTPRODS' and flow_code = 'TOTIMPSB' then value end) as product_imports_kbd,
    max(case when product_code = 'TOTPRODS' and flow_code = 'TOTEXPSB' then value end) as product_exports_kbd,
    max(case when product_code = 'GASDIES'  and flow_code = 'TOTDEMO'  then value end) as diesel_demand_kbd,
    max(case when product_code = 'JETKERO'  and flow_code = 'TOTDEMO'  then value end) as jet_demand_kbd,
    max(case when product_code = 'GASOLINE' and flow_code = 'TOTDEMO'  then value end) as gasoline_demand_kbd,
    max(case when product_code = 'RESFUEL'  and flow_code = 'TOTDEMO'  then value end) as fuel_oil_demand_kbd,
    max(case when product_code = 'CRUDEOIL' and flow_code = 'CLOSTLV'  then value end) as crude_stocks_kbbl,
    max(case when product_code = 'TOTPRODS' and flow_code = 'CLOSTLV'  then value end) as product_stocks_kbbl
from b
group by country_code, period_date
"""

# 1) dbt model file (the 21:00 job's dbt_build makes it official and runs the tests)
with open(DBT_FILE, "w") as f:
    f.write(DBT_SQL)
log("dbt file written: fct_country_balance_monthly.sql (+ fuel_oil_demand_kbd)")

# 2) Rebuild the gold table now on notebook compute, same SQL as dbt
COMPILED = DBT_SQL.replace("{{ ref('fct_oil_balance_monthly') }}", "oil_shock.gold.fct_oil_balance_monthly")
spark.sql("CREATE OR REPLACE TABLE oil_shock.gold.fct_country_balance_monthly AS " + COMPILED)
log("gold.fct_country_balance_monthly rebuilt")

# 3) Checks
t = spark.table("oil_shock.gold.fct_country_balance_monthly")
dup = t.groupBy("row_key").count().where("count > 1").count()
log(f"rows {t.count():,} | duplicate row_key {dup} | columns include fuel_oil_demand_kbd: {'fuel_oil_demand_kbd' in t.columns}")

cov = spark.sql("""
  SELECT date_format(period_date,'yyyy-MM') AS month,
         count(fuel_oil_demand_kbd) AS countries_fo,
         count(diesel_demand_kbd)   AS countries_diesel,
         round(sum(fuel_oil_demand_kbd)/1000, 2) AS fuel_oil_mbd
  FROM oil_shock.gold.fct_country_balance_monthly
  WHERE period_date >= '2026-01-01'
  GROUP BY 1 ORDER BY 1""")
print("\nCoverage 2026 (reporting countries):")
cov.show(20, truncate=False)

# Same-country comparison: countries reporting fuel oil in every month 2025-01..2026-08
cmp = spark.sql("""
  WITH x AS (
    SELECT country_code, period_date, fuel_oil_demand_kbd v
    FROM oil_shock.gold.fct_country_balance_monthly
    WHERE period_date BETWEEN '2025-01-01' AND '2026-08-01' AND fuel_oil_demand_kbd IS NOT NULL),
  ctry AS (SELECT country_code FROM x GROUP BY 1 HAVING count(*) = 20)
  SELECT x.country_code,
         round(avg(CASE WHEN year(period_date)=2025 THEN v END)) AS avg_2025_kbd,
         round(avg(CASE WHEN period_date BETWEEN '2026-03-01' AND '2026-08-01' THEN v END)) AS mar_aug_2026_kbd
  FROM x JOIN ctry USING (country_code)
  GROUP BY 1""")
tot = cmp.selectExpr("count(*) n", "sum(avg_2025_kbd) a", "sum(mar_aug_2026_kbd) b").first()
print(f"Same {tot.n} countries, fuel oil demand: 2025 avg {tot.a/1000:.2f} mb/d -> Mar-Aug 2026 {tot.b/1000:.2f} mb/d ({(tot.b/tot.a-1)*100:+.1f}%)")
print("Top 8 by 2025 volume:")
cmp.orderBy("avg_2025_kbd", ascending=False).withColumn(
    "chg_pct", (cmp.mar_aug_2026_kbd / cmp.avg_2025_kbd - 1) * 100).show(8)

log("DONE")